# Retrain the ui2compose detector on an APP-LEVEL split of ReDraw (SoftwareX revision)

Reviewer 1 asked: *"re-split by app (package name) and report Table 1 again"*. This notebook does exactly that, from the raw ReDraw download to a zip for the laptop, and it runs by itself after you allow Google Drive access.

**What it does**

1. Downloads ReDraw (`ReDraw-Final-Google-Play-Dataset.tar.gz`, 11.6 GB, Zenodo record 2530277). The download resumes after interruptions, and the MD5 is checked.
2. Builds the set of 18,694 labelled screens with the **unchanged v1.1.0 code** (`parse_redraw` → `build_dataset`, cloned from GitHub at tag `v1.1.0`, commit and file hashes pinned).
3. Makes the **app-level split** with `app_split --resplit` from v1.2.0. The v1.2.0 code is embedded in the notebook because it is not on GitHub yet, and every file is checked by SHA-256.
   * A group is a set of screens linked by the same package, an identical MD5 or a dHash distance of at most 4.
   * Groups are assigned 80/10/10 by screens, with multi-label stratification and seed 42.
   * The split is audited independently: no group, package, MD5 or near-duplicate may appear in two splits.
   * `build_dataset --split-by app` cross-checks the split, and its split fingerprints are stored on Drive.
4. Trains **YOLOv8s from COCO `yolov8s.pt`**, first `baseline`, then `oversample_5k`. Both use `train.py` and the repository's `configs/train_*.yaml` with unchanged hyperparameters: AdamW, lr0 0.008, 60 epochs, batch 8, nbs 64, imgsz 640, rect, mosaic 0, seed 42, deterministic.
   * For `oversample_5k`, the v1.2.0 oversampling (8 rare classes, target 5,000) is applied to the **train split only**.
   * The screenshots are pre-resized to 640 px with exactly the resize Ultralytics would apply, so the network input is identical (this is checked).
   * `last.pt` goes to Google Drive after every epoch. If Colab gives you a new machine, *Run all* resumes from there.
5. Evaluates each run on the **new val and test splits** with the evaluation settings of the repository: conf 0.001, NMS IoU 0.7, imgsz 640. It then runs `ui2compose-evaluate-detector`, which adds bootstrap confidence intervals over app groups and the paired baseline vs. oversample_5k difference, and writes **Table 1**.
6. Packs the val + test images, all labels, the val + test raw XMLs with `mapping.csv`, the split manifest, the metrics, both `best.pt` and the provenance into one zip on Drive, with its SHA-256.

**Runtime (estimates; the notebook prints a measured estimate after the first training epoch)**

| part | A100 | L4 |
|---|---|---|
| download (1.4–2.3 MB/s measured from Zenodo) + unpacking + preprocessing + split | 2–3 h | 2–3 h |
| training `baseline` (~15,000 images per epoch, 60 epochs) | 2–3.5 h | 5–7 h |
| training `oversample_5k` (~35,000–38,000 images per epoch) | 5–8 h | 12–16 h |
| evaluation, evaluate_detector, zip | ~0.5 h | ~0.7 h |
| **total** | **about 10–15 h** | **about 20–27 h** |

The estimates assume about 30 T4-hours for one oversample_5k training run (the earlier experience with this repository), an A100 about 4–6 times and an L4 about 2–2.5 times as fast as a T4, and the 2 data-loader workers of the configuration. Early stopping (patience 20) can shorten a run.

**What is saved on Google Drive** (`MyDrive/ui2compose_revision/retrain_app_split/`, about 3 GB in total; the notebook checks the free space):
* `REPORT.txt`: PASS/FAIL checks and key numbers, updated after every step;
* `ui2compose_redraw_appsplit_v1.2.0.zip` (~2.3 GB) and its `.sha256`;
* `metrics/`: metrics per run and split, `evaluate_detector_*.json`, `table1_app_split.md`;
* `runs/<run>_app/`: checkpoints for resuming, `progress.json` (epoch times, ETA) and `final/` (best.pt, last.pt, results.csv, args.yaml);
* `markers/`, `state/`: what is finished (used by *Run all* to skip finished steps);
* `ui2compose_retrain_diagnostics.zip`: a small zip of logs, for diagnosis.

**How to start it**

1. Open [colab.research.google.com](https://colab.research.google.com), choose *File → Upload notebook* and select `retrain_app_split_colab.ipynb`.
2. Choose *Runtime → Change runtime type*. Set the hardware accelerator to **A100 GPU** (or **L4 GPU** if A100 is not available) and switch **High-RAM** on, then *Save*. The notebook asks for this by itself; check it anyway.
3. Choose *Runtime → Run all*. Colab warns that the notebook was not written by Google; choose **Run anyway**.
4. **Within the first minute**, a pop-up asks for access to Google Drive. Choose the Google account that holds `MyDrive/ui2compose_revision/` and allow access.
5. Check the banner of Step 2: it must show **GPU: NVIDIA A100…** (or L4). On a CPU runtime the notebook stops before training and tells you what to change.

**What to watch**
* Step 3: download progress every minute (speed and ETA).
* Step 5: the table **5 app-level split**. Every line must be PASS or INFO.
* Step 6: the line **TIME ESTIMATE after the first epoch** (roughly 2–4 min per epoch for `baseline` and 5–8 min for `oversample_5k` on an A100). Please note it. After that, one line per epoch with the val mAP and the ETA. The same numbers are in `runs/<run>_app/progress.json` on Drive.
* Colab Pro+ keeps running in the background when you close the tab. If you keep it open, answer any *"Are you still there?"* dialog.

**If the runtime disconnects or Colab gives you a new machine:** choose *Runtime → Run all* again (same runtime type). Finished steps are skipped (markers on Drive). On a new machine, the data is downloaded and rebuilt again (2–3 h). The split is checked against the fingerprints on Drive, and training resumes from the last checkpoint on Drive.

**What to send back:** `REPORT.txt` and `metrics/table1_app_split.md`. If something says FAIL, also send `ui2compose_retrain_diagnostics.zip`. Download the zip to the laptop; Drive warns that it cannot scan such a large file for viruses, so choose *Download anyway*.

## Step 1 – Settings, reference values, embedded code, helpers

You normally do **not** change anything here.
* `DRY_RUN`: for developers only. It runs a local test on a tiny synthetic archive (CPU, 1–2 epochs). Leave it unticked.
* `FORCE`: continue after a FAIL. Tick it only if you were told to.
* `RUNS`: the two training runs, `baseline` first, then `oversample_5k`.
* `DRIVE_CKPT_EVERY_EPOCHS = 1`: `last.pt` (~67 MB with optimizer state) is copied to Drive after every epoch. It is copied into two alternating slot files with SHA-256, so an interrupted copy never destroys the last good checkpoint. While Drive has less than `DRIVE_RESERVE_GB` free, the copies pause, so the space for the zip is kept.

Step 1c unpacks the embedded v1.2.0 code: `app_split.py`, `build_dataset.py`, `oversample_dataset.py`, `train.py`, `evaluate_detector.py`, `ap.py`, the configs, and the reference results. It prints the SHA-256 of every file and stops if one differs.

In [ ]:
#@title Step 1 - Settings (the only cell you might edit)
DRY_RUN = False  #@param {type:"boolean"}
FORCE = False    #@param {type:"boolean"}

WORK = '/content/work'                                                         # local disk of the Colab machine
OUT_DRIVE_DIR = '/content/drive/MyDrive/ui2compose_revision/retrain_app_split'  # checkpoints, metrics, zip, REPORT.txt
RUNS = ['baseline', 'oversample_5k']   # trained one after the other, in this order
SEED = 42                              # app_split --resplit seed (= build_dataset SEED); training seed is in the configs

REPO_URL = 'https://github.com/gbilgeturk/ui2compose'
REPO_TAG = 'v1.1.0'                    # parse_redraw / build_dataset of the thesis (defines the 18,694 screens)
ZENODO_RECORD = '2530277'                                # "The ReDraw Dataset" (Moran et al.)
ZENODO_FILE = 'ReDraw-Final-Google-Play-Dataset.tar.gz'  # 11.58 GB: full screenshots + uiautomator XML

ULTRALYTICS_VERSION = '8.4.24'         # version of the published results (configs/train_*.yaml: ultralytics_version)
OVERSAMPLE_TARGET = 5000               # oversample_dataset.py --target-min of the released checkpoint
NEAR_DUP_DIST = 4                      # app_split grouping: dHash Hamming distance <= 4 (v1.2.0 default)
# Evaluation = the repository's evaluate_detector settings (Ultralytics validation defaults); batch 16 = training-time
# validation batch (8 x 2); rect batching as YOLO.val() always does.
VAL_SETTINGS = dict(imgsz=640, batch=16, conf=0.001, iou=0.7, max_det=300, rect=True)
ALSO_RUN_FP16_ON_GPU = True            # training-time validation runs in FP16 (AMP); also evaluate that way
RUN_EVALUATE_DETECTOR = True           # also run ui2compose-evaluate-detector (bootstrap CIs, paired difference)
BOOTSTRAP = 1000                       # bootstrap resamples of evaluate_detector (images and app groups)
CROSS_CHECK_BUILD_DATASET = True       # rebuild the split with build_dataset.py --split-by app; must be identical

CACHE_MODE = 'auto'                    # 'auto': 'ram' if the decoded images fit in RAM, else 'disk' (or 'ram'/'disk')
DRIVE_CKPT_EVERY_EPOCHS = 1            # copy last.pt (+ best.pt when it improves) to Drive every N epochs
DRIVE_MIN_FREE_GB = 4.0                # Drive space needed at the start (zip ~2.3 GB + checkpoints ~0.7 GB + margin)
DRIVE_RESERVE_GB = 3.0                 # per-epoch checkpoint copies stop while Drive has less than this free
MIN_FREE_GB = 60                       # local disk needed on the Colab machine
MAX_AUTO_RESUME = 3                    # automatic restarts of a training run that crashed (from its last.pt)
VERIFY_DRIVE_COPY = True               # read the zip back from Drive and compare its SHA-256

import os, json as _json
DRY = {}
if DRY_RUN:  # local test on a synthetic mini-ReDraw: no Google Drive, CPU, tiny images, 1-2 epochs
    _dry = os.environ.get('UI2C_DRYRUN_DIR', os.path.abspath('ui2c_dryrun'))
    WORK = os.path.join(_dry, 'work')
    OUT_DRIVE_DIR = os.path.join(_dry, 'drive', 'MyDrive', 'ui2compose_revision', 'retrain_app_split')
    REPO_URL = os.environ.get('UI2C_DRYRUN_REPO_URL', REPO_URL)
    DRY = {'record_json': os.environ.get('UI2C_DRYRUN_RECORD_JSON'),   # Zenodo-API look-alike (file:// link)
           'imgsz': int(os.environ.get('UI2C_DRYRUN_IMGSZ', '64')),
           'epochs': _json.loads(os.environ.get('UI2C_DRYRUN_EPOCHS', '{}')),       # {run: epochs}, default 1
           'crash': _json.loads(os.environ.get('UI2C_DRYRUN_CRASH', '{}')),         # {run: {after_epoch, mode}}
           'yolov8s': os.environ.get('UI2C_DRYRUN_YOLOV8S'),                        # local yolov8s.pt (no download)
           'workers': int(os.environ.get('UI2C_DRYRUN_WORKERS', '0'))}
    VAL_SETTINGS = dict(VAL_SETTINGS, imgsz=DRY['imgsz'])
    BOOTSTRAP = int(os.environ.get('UI2C_DRYRUN_BOOTSTRAP', '20'))
    MIN_FREE_GB, DRIVE_MIN_FREE_GB, DRIVE_RESERVE_GB = 1, 0.1, 0.05
print(f'DRY_RUN={DRY_RUN}  FORCE={FORCE}\nWORK={WORK}\nOUT_DRIVE_DIR={OUT_DRIVE_DIR}\nRUNS={RUNS}  SEED={SEED}\n'
      f'parse_redraw/build_dataset: {REPO_URL} @ {REPO_TAG};  app_split/oversample/train/evaluate: embedded v1.2.0')

In [ ]:
#@title Step 1b - Reference values (do not edit)
EXPECTED = {
    'total_pairs': 18694,        # build_dataset v1.1.0 "Total pairs" = labelled screens (thesis Sect. 3.2.3)
    'label_files': 18694,        # parse_redraw v1.1.0 "label files written" (Colab run of 2026-10-20)
    'raw_pairs': 19786,          # parse_redraw v1.1.0 "Total pairs" (XML + screenshot), same run
    'app_folders_in_archive': 6526,
}
# Hyperparameters that must be identical to the released checkpoint (results/train_args_oversample_5k.json);
# checked on the arguments train.py resolves and again on the args.yaml Ultralytics writes.
EXPECTED_TRAIN_ARGS = {'optimizer': 'AdamW', 'lr0': 0.008, 'lrf': 0.01, 'epochs': 60, 'batch': 8, 'nbs': 64,
                       'imgsz': 640, 'rect': True, 'mosaic': 0.0, 'seed': 42, 'deterministic': True,
                       'amp': True, 'pretrained': True, 'patience': 20, 'workers': 2, 'cos_lr': False,
                       'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'fliplr': 0.5,
                       'scale': 0.5, 'translate': 0.1, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
                       'mixup': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10, 'iou': 0.7, 'max_det': 300}
# Keys that legitimately differ from the released checkpoint's train_args: paths, the resume bookkeeping of that
# run (it was resumed from /content/last_os5k.pt), and the cache mode (see Step 6).
TRAIN_ARGS_MAY_DIFFER = ('model', 'data', 'project', 'name', 'resume', 'cache', 'device', 'save_dir')
INIT_WEIGHTS = 'yolov8s.pt'      # COCO-pretrained YOLOv8s (Ultralytics release asset)
INIT_WEIGHTS_SHA256 = '1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a'  # copy used for the thesis audit

PINNED = {'commit': {'v1.1.0': '865bc2e5b110fd3d3f60a8fc62bdeca772f4d616'},
          'files': {'v1.1.0': {
              'src/ui2compose/preprocess/parse_redraw.py': '97e5b1644010095fe3d115ef5939b3e0cf12d1e7a34a2da21548232a10170e16',
              'src/ui2compose/preprocess/build_dataset.py': '49f3d54849cdacb1a431e3b59ccd7647f908b921094dcc9932f996898ec32409',
              'configs/dataset.yaml': '134ed8a6fcc304d7b8fa3922fb1277a2df8596bd3348b33286302b9ee0fc2d6a'}},
          'parse_redraw_defaults': {'min_area': 5e-05, 'dedup_iou': 0.85, 'drop_container': True, 'limit': 0}}
RARE_CLASS_NAMES = ['CheckBox', 'ProgressBar', 'RadioButton', 'RecyclerView', 'SeekBar', 'Spinner', 'Switch', 'WebView']
print('reference values loaded')

In [ ]:
#@title Step 1c - Embedded ui2compose v1.2.0 code (not on GitHub yet; just run it)
# A tar.gz (base64) of the v1.2.0 files this notebook runs. Every file is checked against the SHA-256 below,
# which are the hashes of the reviewed working tree; the same table is printed and written to REPORT.txt.
import base64, hashlib, io, os, tarfile
from pathlib import Path

EMBED_TAR_SHA256 = '8c31f73e8c7c7cd7993684a9e61e1b9cea5312b73bfe5cc98f146eaba71d3a3e'
EMBED_SHA256 = {
    'src/ui2compose/__init__.py': 'e7b85137629f270c61050aa0447fd3799d8624978087ea316e9c4be5a35ab57a',
    'src/ui2compose/preprocess/__init__.py': '8fa46c0f1d1abeffd0b0ea557d2b0260339c3a922868e9a6f90578ec90e239b5',
    'src/ui2compose/preprocess/app_split.py': '60b693d1719f5241cf2f0bb06a4e229e8c6c44770ff7987f4fcf10736f2b388b',
    'src/ui2compose/preprocess/build_dataset.py': 'b5ef154cf083d7a35e22b5c2aaa14d73f4fbf6f6f9ab74dd739bae2d7dd364ef',
    'src/ui2compose/preprocess/oversample_dataset.py': 'ec76a7cb814fcca977fd7109aceb7643c9c1fa516597f970c2fed79472630243',
    'src/ui2compose/train.py': '82cd9e0479421529c3100ee526c7c3aacc396e1421ceb02a7759aff779fe531f',
    'src/ui2compose/evaluation/__init__.py': '63a943454abb1630694380c26dcc41276ec9b36d8ce8cc17a9de5ea1524c8c73',
    'src/ui2compose/evaluation/ap.py': '6b65b93ee89e5bb55bdc514b618c3abd88f973884be16ae02f76013b0edc2351',
    'src/ui2compose/evaluation/evaluate_detector.py': '6c3a9a3b2c89d273ae03ee984df7a0deba086bf2cda8cb3d003d5e687eb4d459',
    'configs/dataset.yaml': '134ed8a6fcc304d7b8fa3922fb1277a2df8596bd3348b33286302b9ee0fc2d6a',
    'configs/train_baseline.yaml': '3977072420d57f2bcf396e41e6a57265a10281bd65dd1d1938e44d32bb84efa5',
    'configs/train_oversample_5k.yaml': '2924b33c1a2a327710d7362f34324c005baa2ddf29a070f1789e047c0f44ec74',
    'results/train_args_oversample_5k.json': 'fb1d99e52b879df6853cdd07d1e9866ddb9ac3754bde1be8086aea02abbf6a07',
    'results/perclass_ap_results.json': 'bfe0b8b9876a875f551cb3285664df774819ceef3245d50c045eae49065e548b',
    'results/imgsz_640_metrics.json': '3065f0e068a53646252bf3b8663a37b0ece3b827b7e04d938bcc20c193445494',
    'results/oversample_5k_metrics.json': 'd7c1d3c1a739a1c5f310c28c2a9674dd1569a6f984510a06a05add71f155c9d2',
}
EMBED_B64 = '''
...
'''
_raw = base64.b64decode(''.join(EMBED_B64.split()))
_got = hashlib.sha256(_raw).hexdigest()
if _got != EMBED_TAR_SHA256:
    raise RuntimeError(f'embedded archive has SHA-256 {_got}, expected {EMBED_TAR_SHA256}: the cell was modified. '
                       'Upload the original notebook again.')
CODE = Path(WORK) / 'ui2compose_v1.2.0'          # PYTHONPATH=CODE/src for the v1.2.0 modules
with tarfile.open(fileobj=io.BytesIO(_raw), mode='r:gz') as _tf:
    _names = sorted(m.name for m in _tf.getmembers() if m.isfile())
    if sorted(EMBED_SHA256) != _names:
        raise RuntimeError(f'embedded archive lists {_names}, expected {sorted(EMBED_SHA256)}')
    for _m in _tf.getmembers():
        if _m.isfile():
            _dest = CODE / _m.name
            _dest.parent.mkdir(parents=True, exist_ok=True)
            _dest.write_bytes(_tf.extractfile(_m).read())
EMBED_CHECK = {}
for _rel, _h in EMBED_SHA256.items():
    _hh = hashlib.sha256((CODE / _rel).read_bytes()).hexdigest()
    EMBED_CHECK[_rel] = {'sha256': _hh, 'ok': _hh == _h}
    print(f"{'OK  ' if _hh == _h else 'BAD '} {_hh}  {_rel}")
if not all(v['ok'] for v in EMBED_CHECK.values()):
    raise RuntimeError('an embedded file does not match its SHA-256: upload the original notebook again')
print(f'embedded v1.2.0 code extracted to {CODE} (archive sha256 {EMBED_TAR_SHA256[:16]}...)')

In [ ]:
#@title Step 1d - Helper functions (just run it; no need to read)
import csv, datetime, hashlib, io, json, math, os, platform, random, re, shlex, shutil, signal, subprocess, sys
import threading, time, zipfile
from collections import Counter, defaultdict
from pathlib import Path

WORK = Path(WORK)
# Local disk (lost when Colab gives you a new machine): data and everything rebuilt from it.
D = {k: WORK / k for k in ('markers', 'logs', 'download', 'raw', 'repos', 'data', 'tools', 'out', 'runs',
                           'metrics', 'cfg', 'weights')}
for _p in D.values():
    _p.mkdir(parents=True, exist_ok=True)
# Google Drive (survives a new machine): markers of finished long steps, state, checkpoints, metrics, zip, report.
DD = Path(OUT_DRIVE_DIR)
DM, DS, DRUNS, DMET = DD / 'markers', DD / 'state', DD / 'runs', DD / 'metrics'
IMG_EXTS = ('.png', '.jpg', '.jpeg')
SPLITS = ('train', 'val', 'test')
STEM_RE = re.compile(r'^(?P<app>.+-screens)_screenshot_(?P<n>\d+)$')   # build_dataset output names
DIAG_ZIP = 'ui2compose_retrain_diagnostics.zip'
SUBENV = dict(os.environ, PYTHONPATH=str(CODE / 'src') + (os.pathsep + os.environ['PYTHONPATH']
                                                          if os.environ.get('PYTHONPATH') else ''),
              PYTHONUNBUFFERED='1')


class PipelineStop(Exception):
    """Stops 'Run all' with a short message instead of a long traceback."""
    def _render_traceback_(self):
        bar = '=' * 78
        return [bar, 'STOPPED: ' + str(self), bar]


def now():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')


def jdump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')   # small files: overwrite in place


def jload(path, default=None):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except (FileNotFoundError, json.JSONDecodeError):
        return default


# Local markers: a step that rebuilt local data writes WORK/markers/<name>.done (lost with the machine).
def done(name):
    return (D['markers'] / f'{name}.done').exists()


def mark(name, info=None):
    jdump({'finished': now(), **(info or {})}, D['markers'] / f'{name}.done')


def marker(name):
    return jload(D['markers'] / f'{name}.done', {})


# Drive markers: long steps whose results live on Drive (training, metrics, zip) are never repeated.
def ddone(name):
    return (DM / f'{name}.done').exists()


def dmark(name, info=None):
    jdump({'finished': now(), **(info or {})}, DM / f'{name}.done')


def dmarker(name):
    return jload(DM / f'{name}.done', {})


def save_state(name, obj):
    jdump(obj, DS / f'{name}.json')


def load_state(name, default=None):
    return jload(DS / f'{name}.json', default)


def human(n):
    n = float(n)
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if abs(n) < 1024 or unit == 'TB':
            return f'{n:.0f} {unit}' if unit == 'B' else f'{n:.2f} {unit}'
        n /= 1024


def free_gb(path):
    return shutil.disk_usage(path).free / 1e9


def file_hash(path, algo='sha256', progress_every=0):
    h, n, t0, last = hashlib.new(algo), 0, time.time(), time.time()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 << 20), b''):
            h.update(chunk)
            n += len(chunk)
            if progress_every and time.time() - last > progress_every:
                last = time.time()
                print(f'  {algo}: {human(n)} read ({n / max(last - t0, 1e-9) / 1e6:.0f} MB/s)', flush=True)
    return h.hexdigest()


def qcmd(cmd):
    return ' '.join(shlex.quote(str(c)) for c in cmd)


def kill_stale(substr, programs):
    """Stops processes of an earlier, interrupted run (e.g. a curl or a training that survived a kernel restart).
    programs: program-name prefixes."""
    try:
        out = subprocess.run(['ps', '-ww', '-eo', 'pid=,args='], capture_output=True, text=True, timeout=30).stdout
    except Exception:
        return []
    killed = []
    for line in out.splitlines():
        parts = line.strip().split(None, 1)
        if len(parts) != 2 or not parts[0].isdigit() or int(parts[0]) == os.getpid():
            continue
        prog = os.path.basename(parts[1].split()[0])
        if str(substr) in parts[1] and any(prog.startswith(p) for p in programs):
            try:
                os.kill(int(parts[0]), signal.SIGTERM)
                killed.append(int(parts[0]))
            except OSError:
                pass
    if killed:
        time.sleep(2)
        for pid in killed:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
        print(f'stopped {len(killed)} leftover process(es) of an earlier run: {killed}', flush=True)
    return killed


_ANSI = re.compile(r'\x1b\[[0-9;?]*[A-Za-z]')


def run(cmd, cwd=None, log=None, check=True, echo=True, quiet=False, env=None, progress_every=60):
    """Runs a command, streams its output, appends it to logs/<log>; raises PipelineStop on failure.
    Progress bars ('\\r' updates) are shown at most every progress_every seconds. If the cell is interrupted,
    the command is killed too (no process keeps running in the background)."""
    cmd = [str(c) for c in cmd]
    if not quiet:
        print('$ ' + qcmd(cmd), flush=True)
    t0, lines = time.time(), []
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env or SUBENV)
    logf = open(D['logs'] / log, 'a', encoding='utf-8') if log else None
    if logf:
        logf.write(f'\n### {now()}  cwd={cwd}\n$ {qcmd(cmd)}\n')
    buf, last_progress, pending = b'', 0.0, None

    def emit(text, newline):
        nonlocal last_progress, pending
        text = _ANSI.sub('', text)
        if newline:
            lines.append(text + '\n')
            if logf:
                logf.write(text + '\n')
            if echo:
                print(text, flush=True)
            pending = None
        elif text.strip():
            pending = text
            if echo and time.time() - last_progress >= progress_every:
                last_progress = time.time()
                print(text.strip(), flush=True)

    try:
        while True:
            chunk = p.stdout.read1(65536) if hasattr(p.stdout, 'read1') else p.stdout.read(4096)
            if not chunk:
                break
            buf += chunk
            while True:
                m = re.search(rb'[\r\n]', buf)
                if not m:
                    break
                piece, sep, buf = buf[:m.start()], buf[m.start():m.start() + 1], buf[m.end():]
                emit(piece.decode('utf-8', 'replace'), sep == b'\n')
        if buf:
            emit(buf.decode('utf-8', 'replace'), True)
        rc = p.wait()
    except BaseException:
        p.kill()
        p.wait()
        if logf:
            logf.write('### INTERRUPTED (killed)\n')
            logf.close()
        raise
    if logf:
        logf.write(f'### exit {rc} after {time.time() - t0:.0f} s\n')
        logf.close()
    out = ''.join(lines)
    if check and rc != 0:
        raise PipelineStop(f'command failed with exit code {rc}: {qcmd(cmd)[:300]}'
                           + (f'  (log: {D["logs"] / log})' if log else ''))
    return rc, out


# ---------------- PASS/FAIL tables ----------------
def fmt(v):
    if isinstance(v, bool) or v is None:
        return str(v)
    if isinstance(v, float):
        return f'{v:.5f}'
    if isinstance(v, int):
        return f'{v:,}'
    return str(v)


def check_row(check, expected, got, ok=None, status=None, note=''):
    if status is None:
        status = 'PASS' if (ok if ok is not None else expected == got) else 'FAIL'
    return {'check': check, 'expected': expected, 'got': got, 'status': status, 'note': note}


def table_lines(title, rows):
    out = [title, '-' * len(title)]
    w = max([len(str(r['check'])) for r in rows] + [8])
    for r in rows:
        line = f"{r['status']:<5} {str(r['check']):<{w}}  expected {fmt(r['expected'])}  got {fmt(r['got'])}"
        if r.get('note'):
            line += f"  [{r['note']}]"
        out.append(line)
    return out


def set_checks(section, rows, show=True):
    allc = load_state('checks', {})
    allc[section] = rows
    save_state('checks', allc)
    if show:
        print('\n' + '\n'.join(table_lines(section, rows)))
    return all(r['status'] in ('PASS', 'INFO', 'WARN') for r in rows)


def require(ok, what):
    """Stops on a FAIL unless FORCE is ticked (then it continues and the report says FORCED)."""
    if ok:
        return
    write_report()
    if FORCE:
        print(f'WARNING: {what} FAILED, continuing because FORCE is ticked')
        return
    raise PipelineStop(f'{what} FAILED (see the table above and REPORT.txt). Nothing was trained on a wrong split. '
                       'Send REPORT.txt back; tick FORCE only if you were told to.')


def banner(text, ok=True):
    bar = ('=' if ok else '#') * 78
    print(f'\n{bar}\n{text}\n{bar}\n', flush=True)


# ---------------- Google Drive ----------------
def drive_alive():
    """True if OUT_DRIVE_DIR is on a working Drive mount (a test file can be written)."""
    if DRY_RUN:
        DD.mkdir(parents=True, exist_ok=True)
        return True
    try:
        if not os.path.ismount('/content/drive'):
            return False   # never write below /content/drive without a mount (a later mount would fail)
        DD.mkdir(parents=True, exist_ok=True)
        (DD / '.ui2compose_write_test').write_text(now())   # overwritten, never deleted
        return True
    except OSError:
        return False


def mount_drive(force=False):
    """Connects Google Drive (Colab shows a pop-up). Raises PipelineStop with instructions if that fails."""
    if DRY_RUN:
        print(f'DRY_RUN: Google Drive is not mounted; "Drive" output goes to {DD}')
    elif not force and drive_alive():
        print(f'Google Drive is connected; results go to {DD}')
    else:
        print('Connecting Google Drive: a pop-up opens. Choose your Google account and allow access '
              '(within 2 minutes).', flush=True)
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=force or os.path.ismount('/content/drive'))
        except Exception as e:
            raise PipelineStop(f'Google Drive could not be connected ({type(e).__name__}: {e}). Run this cell again '
                               '(or Runtime -> Run all) and allow access in the pop-up. Finished steps are skipped.')
        if not drive_alive():
            raise PipelineStop(f'Google Drive is mounted, but {DD} cannot be written. Check that your Drive '
                               'is not full, then run this cell again.')
        print(f'Google Drive connected; results go to {DD}')
    for p in (DD, DM, DS, DRUNS, DMET):
        p.mkdir(parents=True, exist_ok=True)


def drive_free_gb():
    """Free space on Google Drive in GB as the mount reports it (None if it reports nothing useful)."""
    try:
        u = shutil.disk_usage(DD if DRY_RUN else '/content/drive')
        return u.free / 1e9 if u.total > 0 else None
    except OSError:
        return None


def drive_used_by_us():
    return sum(p.stat().st_size for p in DD.rglob('*') if p.is_file()) if DD.exists() else 0


def copy_to_drive(src, dest, verify=False):
    """Copies a file to Drive (dest: a path below DD). The destination is overwritten in place, never deleted
    first (deleted Drive files pile up in the Drive bin and keep using space). Returns (dest, sha256_ok)."""
    dest = Path(dest)
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dest)
        ok = (file_hash(dest) == file_hash(src)) if verify else None
    except OSError as e:
        print(f'WARNING: copying {Path(src).name} to Google Drive failed ({e}); the file stays at {src}')
        return None, None
    return dest, ok


def drive_flush():
    """Writes everything still cached to Google Drive and unmounts it (Colab only)."""
    if DRY_RUN or not os.path.ismount('/content/drive'):
        return
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        print('Google Drive flushed and unmounted: everything written so far is on your Drive.')
    except Exception as e:
        print(f'WARNING: flushing Google Drive failed ({e}); wait a few minutes before closing the tab.')


# ---------------- what still has to be done ----------------
def train_key(run_name):
    return f'train_{run_name}'


def eval_key(run_name):
    return f'eval_{run_name}'


def remaining_work():
    """Long steps whose results are not on Drive yet. Empty: nothing needs the ReDraw data any more."""
    todo = [k for r in RUNS for k in (train_key(r), eval_key(r)) if not ddone(k)]
    if RUN_EVALUATE_DETECTOR and not ddone('evaluate_detector'):
        todo.append('evaluate_detector')
    if not ddone('package'):
        todo.append('package')
    return todo


def training_left():
    return [r for r in RUNS if not ddone(train_key(r))]


# ---------------- repository and preprocessing ----------------
def repo_dir(tag):
    return D['repos'] / f'ui2compose_{tag}'


def clone_repo(tag):
    """Clones REPO_URL at a tag (blobs > 5 MB, i.e. the weights, are not fetched) and checks commit + file hashes."""
    dest, key = repo_dir(tag), f'clone_{tag}'
    if done(key) and dest.exists():
        info = marker(key)
        print(f'[skip] {tag} already cloned at {dest} (commit {info.get("commit")})')
        return dest, info
    if dest.exists():
        shutil.rmtree(dest)
    run(['git', '-c', 'advice.detachedHead=false', 'clone', '--quiet', '--depth', '1', '--filter=blob:limit=5m',
         '--branch', tag, REPO_URL, dest], log='git.log')
    commit = run(['git', '-C', dest, 'rev-parse', 'HEAD'], echo=False, quiet=True)[1].strip()
    pinned_files = PINNED['files'].get(tag, {})
    files = {rel: (file_hash(dest / rel) if (dest / rel).exists() else None) for rel in pinned_files}
    problems = []
    if PINNED['commit'].get(tag) and commit != PINNED['commit'][tag]:
        problems.append(f'commit {commit} differs from the pinned {PINNED["commit"][tag]}')
    problems += [f'{rel}: SHA-256 {h} differs from the pinned value' for rel, h in files.items()
                 if h != pinned_files[rel]]
    info = {'tag': tag, 'url': REPO_URL, 'commit': commit, 'file_sha256': files, 'problems': problems}
    mark(key, info)
    print(f'{tag}: commit {commit} - ' + ('commit and file hashes match the pinned values' if not problems
                                          else 'WARNING: ' + '; '.join(problems)))
    return dest, info


_ARGS_PROBE = '''
import importlib.util, json, sys
spec = importlib.util.spec_from_file_location("probe_mod", sys.argv[1])
m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
sys.argv = [sys.argv[1]] + sys.argv[2:]
consts = {k: getattr(m, k) for k in ("SEED", "SPLIT") if hasattr(m, k)}
print("ARGS_JSON " + json.dumps({"args": vars(m.parse_args()), "constants": consts}))
'''


def effective_args(script, argv):
    _, out = run([sys.executable, '-c', _ARGS_PROBE, script, *argv], echo=False, quiet=True)
    line = [l for l in out.splitlines() if l.startswith('ARGS_JSON ')][-1]
    return json.loads(line[len('ARGS_JSON '):])


def raw_root():
    info = marker('extract')
    if not info:
        raise PipelineStop('Step 3 (download and unpack) has not finished - run Step 3 first.')
    return Path(info['raw_root'])


# ---------------- dataset helpers ----------------
def read_names(yaml_path):
    import yaml
    data = yaml.safe_load(Path(yaml_path).read_text(encoding='utf-8')) or {}
    names = data.get('names', [])
    return [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)


def stems_sha256(stems):  # identical to app_split split_fingerprints (sorted stems, '\n'-terminated)
    return hashlib.sha256(''.join(f'{s}\n' for s in sorted(stems)).encode('utf-8')).hexdigest()


def list_images(yolo, split):
    d = Path(yolo) / 'images' / split
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS) if d.is_dir() else []


def count_labels(yolo, names, split, stems=None):
    """Instances / images per class of labels/<split> (optionally only the given stems)."""
    inst, with_cls, n, empty = Counter(), Counter(), 0, 0
    for lf in sorted((Path(yolo) / 'labels' / split).glob('*.txt')):
        if stems is not None and lf.stem not in stems:
            continue
        here = Counter()
        for ln in lf.read_text(encoding='utf-8').splitlines():
            parts = ln.split()
            if len(parts) >= 5:
                here[int(float(parts[0]))] += 1
        n += 1
        empty += not here
        inst.update(here)
        with_cls.update(here.keys())
    nm = lambda c: names[c] if c < len(names) else f'class_{c}'
    ids = sorted(set(range(len(names))) | set(inst))
    return {'label_files': n, 'empty_label_files': empty, 'instances': sum(inst.values()),
            'per_class_instances': {nm(c): inst.get(c, 0) for c in ids},
            'per_class_images': {nm(c): with_cls.get(c, 0) for c in ids}}


def class_table(counts, splits=SPLITS):
    names = list(counts[splits[0]]['per_class_instances'])
    lines = [f"{'class':<14}" + ''.join(f'{s + " inst":>15}{s + " img":>13}' for s in splits)]
    for n in names:
        lines.append(f'{n:<14}' + ''.join(f"{counts[s]['per_class_instances'].get(n, 0):>15,}"
                                          f"{counts[s]['per_class_images'].get(n, 0):>13,}" for s in splits))
    lines.append(f"{'TOTAL':<14}" + ''.join(f"{counts[s]['instances']:>15,}{counts[s]['label_files']:>13,}"
                                            for s in splits))
    return lines


# ---------------- REPORT.txt and the diagnostics bundle ----------------
def _verdict(checks, prefix):
    xs = [r['status'] for k, v in checks.items() if k.startswith(prefix) for r in v]
    if not xs:
        return 'not run'
    return 'FAIL' if 'FAIL' in xs else 'PASS (with WARN)' if 'WARN' in xs else 'PASS'


def key_numbers():
    L = []
    env, sp, ds = load_state('env') or {}, load_state('split') or {}, load_state('datasets') or {}
    if env:
        L.append(f"machine: {env.get('gpu') or 'no GPU'}; {env.get('cpus')} CPUs; RAM {env.get('ram_gb')} GB; "
                 f"torch {env.get('torch')}; ultralytics {env.get('ultralytics')}")
    if sp:
        s = sp['sizes']
        L.append(f"app-level split (seed {sp.get('seed')}): screens train/val/test {s['train']:,} / {s['val']:,} / "
                 f"{s['test']:,} of {sum(s.values()):,}; apps {sp['apps']['train']:,} / {sp['apps']['val']:,} / "
                 f"{sp['apps']['test']:,}; groups {sp['groups']['train']:,} / {sp['groups']['val']:,} / "
                 f"{sp['groups']['test']:,}; audit {'PASS' if sp.get('audit_ok') else 'FAIL'}")
        L.append(f"instances train/val/test: {sp['instances']['train']:,} / {sp['instances']['val']:,} / "
                 f"{sp['instances']['test']:,}")
        L.append('split fingerprints (sha256 of sorted stems): ' +
                 ', '.join(f"{k} {v[:16]}..." for k, v in sp['fingerprints'].items()))
    if ds.get('os5k'):
        o = ds['os5k']
        L.append(f"oversample_5k train: {o['train_images']:,} images ({o['copies']:,} copies); rare classes after: "
                 + ', '.join(f'{k} {v:,}' for k, v in o['rare_after'].items()))
    for r in RUNS:
        t = dmarker(train_key(r))
        if t:
            L.append(f"{r}: {t.get('epochs_done')} epochs (best epoch {t.get('best_epoch')}), "
                     f"{t.get('hours_total')} h on {t.get('gpu')}; mean epoch {t.get('mean_epoch_min')} min")
        e = load_state(f'metrics_{r}') or {}
        for k, v in sorted((e.get('values') or {}).items()):
            L.append(f"  {r} {k}: mAP@50 {v['mAP50']:.5f}  mAP@50-95 {v['mAP50_95']:.5f}  P {v['precision']:.4f}  "
                     f"R {v['recall']:.4f}")
    t1 = load_state('table1') or {}
    for line in t1.get('lines', []):
        L.append(line)
    pkg = load_state('package') or {}
    if pkg:
        L.append(f"zip: {pkg.get('name')}  {pkg.get('size_human')}  sha256 {pkg.get('sha256')}")
        L.append(f"zip on Drive: {pkg.get('drive_path') or 'NOT COPIED'}  (read back OK: {pkg.get('drive_sha256_ok')})")
    df = drive_free_gb()
    L.append(f"Drive: {human(drive_used_by_us())} used by this notebook in {DD}; "
             + (f'{df:.1f} GB free' if df is not None else 'free space not reported'))
    return L


def write_report(final=False):
    """Collects every PASS/FAIL table into REPORT.txt (local copy + Drive copy). Returns (text, drive_path)."""
    checks = load_state('checks', {})
    L = ['ui2compose - retraining on an APP-LEVEL split of ReDraw - REPORT',
         f'written: {now()}   DRY_RUN={DRY_RUN}   FORCE={FORCE}', '', 'OVERALL',
         f"  environment, GPU (Step 2):            {_verdict(checks, '2 ')}",
         f"  download + MD5 + unpack (Step 3):     {_verdict(checks, '3 ')}",
         f"  v1.1.0 screens (Step 4):              {_verdict(checks, '4 ')}",
         f"  app-level split + audit (Step 5):     {_verdict(checks, '5 ')}",
         f"  datasets, oversampling (Step 5b):     {_verdict(checks, '5b')}"]
    for r in RUNS:
        L.append(f"  training {r:<14} (Step 6):      {_verdict(checks, '6 ' + r)}"
                 + ('  [done]' if ddone(train_key(r)) else ''))
        L.append(f"  metrics  {r:<14} (Step 6):      {_verdict(checks, '6m ' + r)}")
    L += [f"  evaluate_detector, Table 1 (Step 7): {_verdict(checks, '7 ')}",
          f"  zip on Drive (Step 8):                {_verdict(checks, '8 ')}",
          f"  still to do: {', '.join(remaining_work()) or 'nothing'}", '']
    try:
        kn = key_numbers()
    except Exception as e:   # the report must never stop the notebook
        kn = [f'(key numbers not available: {type(e).__name__}: {e})']
    L += ['KEY NUMBERS (send back the whole file; if that is not possible, copy at least this block)']
    L += ['  ' + x for x in kn] + ['']
    env = load_state('env') or {}
    if env:
        L += ['ENVIRONMENT', '  ' + json.dumps(env, default=str), '']
    L += ['EMBEDDED v1.2.0 FILES (sha256)'] + [f"  {v['sha256']}  {k}" for k, v in EMBED_CHECK.items()] + ['']
    ci = marker(f'clone_{REPO_TAG}') or load_state('clone') or {}
    if ci:
        L.append(f"CODE {REPO_TAG}: commit {ci.get('commit')}  " + ('(pinned hashes OK)' if not ci.get('problems')
                                                                   else 'PROBLEMS: ' + '; '.join(ci['problems'])))
    for key in ('parse', 'build'):
        i = load_state(f'{key}_{REPO_TAG}') or {}
        if i:
            L.append(f"  {key}: {i.get('command')}")
            L.append(f"      -> {i.get('summary')}")
    L.append('')
    order = lambda kv: (kv[0].split()[0].rstrip('bm'), kv[0])
    for sec, rows in sorted(checks.items(), key=order):
        L += table_lines(sec, rows) + ['']
    sp = load_state('split') or {}
    if sp.get('counts'):
        L += ['PER-CLASS COUNTS OF THE APP-LEVEL SPLIT (original screens)'] + class_table(sp['counts']) + ['']
    L.append('final report' if final else 'interim report (later steps not finished yet)')
    text = '\n'.join(L) + '\n'
    (D['out'] / 'REPORT.txt').write_text(text, encoding='utf-8')
    dest = None
    if drive_alive():
        try:
            dest = DD / 'REPORT.txt'
            dest.write_text(text, encoding='utf-8')
        except OSError as e:
            print(f'WARNING: REPORT.txt could not be written to Google Drive ({e}); local copy: {D["out"] / "REPORT.txt"}')
            dest = None
    return text, dest


def write_diagnostics():
    """Small zip (no images, no weights) for diagnosis: REPORT.txt, Drive state/markers, local markers and logs,
    the split manifests and training progress files. Copied to Drive next to REPORT.txt."""
    text, _ = write_report()
    path = D['out'] / DIAG_ZIP
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        zf.writestr('ui2compose_retrain_diagnostics/REPORT.txt', text)
        for base, sub in ((DS, 'drive_state'), (DM, 'drive_markers'), (D['markers'], 'local_markers'),
                          (D['logs'], 'logs'), (D['metrics'], 'metrics')):
            for p in sorted(base.rglob('*')) if base.exists() else []:
                if p.is_file() and p.stat().st_size < 50e6 and not p.name.endswith('.tmp'):
                    zf.write(p, f'ui2compose_retrain_diagnostics/{sub}/{p.relative_to(base).as_posix()}')
        for p in sorted(DRUNS.rglob('*')) if DRUNS.exists() else []:
            if p.is_file() and p.suffix in ('.json', '.csv', '.yaml'):
                zf.write(p, f'ui2compose_retrain_diagnostics/drive_runs/{p.relative_to(DRUNS).as_posix()}')
    dest, _ = copy_to_drive(path, DD / DIAG_ZIP) if drive_alive() else (None, None)
    print(f'diagnostics bundle: {path} ({human(path.stat().st_size)})'
          + (f' -> {dest}' if dest else ' (not copied to Google Drive)'))
    return path, dest


print('helpers ready; local work directory', WORK, '; Drive folder', DD)

## Step 2 – Google Drive, GPU, packages

**Google Drive first:** a few seconds after *Run all*, a pop-up asks for access to your Google Drive. Allow it.
* The notebook stops if Drive has less than about 4 GB free. At the start it needs ~2.3 GB for the zip, ~0.7 GB for checkpoints and a margin.
* It prints the **GPU type** and stops on a CPU runtime, because training needs a GPU.
* It installs `ultralytics==8.4.24`, the version of the published results. pip only adds what is missing, so Colab's `torch` is not changed.
* It records RAM, disk and all versions in `REPORT.txt`.

In [ ]:
#@title Step 2 - Connect Google Drive, check GPU / RAM / disk, install Ultralytics
import importlib.metadata as _md


def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None


# 1) Google Drive first, so that the pop-up appears within seconds of "Run all"
mount_drive()
todo = remaining_work()
drive_free = drive_free_gb()
used = drive_used_by_us()
need_drive = 0.0 if not todo else max(0.3, DRIVE_MIN_FREE_GB - used / 1e9)   # what we already wrote counts
print(f'Google Drive free space: ' + (f'{drive_free:.1f} GB' if drive_free is not None else 'not reported')
      + f'; already written by this notebook: {human(used)}; still needed: about {need_drive:.1f} GB')
if drive_free is not None and drive_free < need_drive and not DRY_RUN:
    raise PipelineStop(f'your Google Drive has only {drive_free:.1f} GB free, but checkpoints, metrics and the zip '
                       f'need about {need_drive:.1f} GB. Free up space on drive.google.com (and empty its bin), '
                       'then choose Run all again.')
print('still to do:', ', '.join(todo) or 'nothing (everything is on Drive)')

# 2) GPU, RAM, local disk, Python
gpu_name, gpu_mem = None, None
try:
    _q = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True, timeout=30).stdout.strip()
    if _q:
        gpu_name, gpu_mem = [x.strip() for x in _q.splitlines()[0].split(',', 1)]
except Exception:
    pass
try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / 1e9, 1)
    ram_avail_gb = round(psutil.virtual_memory().available / 1e9, 1)
except Exception:
    ram_gb = ram_avail_gb = None
free = free_gb(WORK)
gpu_kind = next((k for k in ('A100', 'H100', 'L4', 'T4', 'V100') if gpu_name and k in gpu_name), 'other' if gpu_name else None)
print(f'Python {sys.version.split()[0]} on {platform.platform()}; {os.cpu_count()} CPUs; RAM {ram_gb} GB '
      f'({ram_avail_gb} GB available); free disk at {WORK}: {free:.1f} GB (need >= {MIN_FREE_GB} GB)')
banner(f'GPU: {gpu_name} ({gpu_mem})' if gpu_name else 'GPU: NONE (CPU runtime)', ok=bool(gpu_name))
if not gpu_name and training_left() and not DRY_RUN:
    write_report()
    raise PipelineStop('this is a CPU runtime, and training needs a GPU. Choose Runtime -> Change runtime type -> '
                       'A100 GPU (or L4 GPU) with "High-RAM", then Runtime -> Run all. Nothing was lost: finished '
                       'steps are on Drive.')
if gpu_kind in ('T4', 'V100', 'other') and training_left() and not DRY_RUN:
    print(f'WARNING: {gpu_name} is slow for this job (a T4 needs about 30 h per training run). It works (the runs '
          'resume from Drive across sessions), but an A100 or L4 runtime is strongly recommended.')
if free < MIN_FREE_GB and todo and not (done('extract') and free > 20):
    raise PipelineStop(f'only {free:.1f} GB free disk at {WORK}; this notebook needs {MIN_FREE_GB} GB. '
                       'Use a fresh runtime (Runtime -> Disconnect and delete runtime).')

# 3) ultralytics pinned to the training version (+ the other requirements of the repository, without streamlit).
# pip installs only what is missing: Colab's torch / CUDA build is not touched.
REQUIREMENTS = [f'ultralytics=={ULTRALYTICS_VERSION}', 'opencv-python>=4.8.0', 'Pillow>=10.0.0', 'numpy>=1.24.0',
                'PyYAML>=6.0', 'matplotlib>=3.7.0', 'psutil', 'polars']
torch_before = _ver('torch')
if DRY_RUN:
    print(f'DRY_RUN: not installing anything (ultralytics {_ver("ultralytics")} found)')
elif not done('pip') or _ver('ultralytics') != ULTRALYTICS_VERSION:
    run([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS], log='pip.log')
    mark('pip', {'requirements': REQUIREMENTS})
torch_after = _ver('torch')
import torch
cuda_ok = torch.cuda.is_available()
env = {'python': sys.version.split()[0], 'platform': platform.platform(), 'cpus': os.cpu_count(),
       'gpu': gpu_name, 'gpu_memory': gpu_mem, 'gpu_kind': gpu_kind, 'ram_gb': ram_gb, 'ram_available_gb': ram_avail_gb,
       'free_disk_gb': round(free, 1), 'drive_free_gb': round(drive_free, 1) if drive_free else None,
       'torch': torch_after, 'torch_before_pip': torch_before, 'cuda': torch.version.cuda, 'cuda_available': cuda_ok,
       'cudnn': torch.backends.cudnn.version() if cuda_ok else None,
       'ultralytics': _ver('ultralytics'), 'numpy': _ver('numpy'), 'Pillow': _ver('Pillow') or _ver('pillow'),
       'PyYAML': _ver('PyYAML'), 'opencv': _ver('opencv-python') or _ver('opencv-python-headless'),
       'polars': _ver('polars'), 'git': run(['git', '--version'], echo=False, quiet=True)[1].strip(), 'started': now()}
save_state('env', env)
envs = load_state('env_history', [])
save_state('env_history', envs + [env])   # one entry per machine/session (a new machine adds a line)
rows = [check_row('Google Drive connected', True, drive_alive()),
        check_row('Google Drive free (GB)', f'>= {need_drive:.1f}', round(drive_free, 1) if drive_free is not None
                  else 'not reported', status='PASS' if drive_free is not None and drive_free >= need_drive else 'WARN'),
        check_row('free disk (GB)', f'>= {MIN_FREE_GB}', round(free, 1), ok=free >= MIN_FREE_GB or not todo or done('extract')),
        check_row('ultralytics version', ULTRALYTICS_VERSION, env['ultralytics']),
        check_row('torch unchanged by pip', torch_before, torch_after),
        check_row('GPU (training needs one)', 'A100 / L4', gpu_name or 'none',
                  status='PASS' if gpu_kind in ('A100', 'H100', 'L4') else ('INFO' if DRY_RUN or not training_left()
                                                                            else 'WARN')),
        check_row('torch sees the GPU', bool(gpu_name), cuda_ok, status=None if gpu_name else 'INFO'),
        check_row('RAM (GB)', 'reported', ram_gb, status='INFO', note='Step 6 chooses cache=ram if the images fit')]
set_checks('2 environment', rows)
if env['ultralytics'] != ULTRALYTICS_VERSION:
    raise PipelineStop(f'ultralytics {env["ultralytics"]} is installed, {ULTRALYTICS_VERSION} is needed. '
                       'Choose Runtime -> Restart session, then Run all again.')
if gpu_name and not cuda_ok and not DRY_RUN:
    raise PipelineStop('nvidia-smi shows a GPU but torch cannot use it. Choose Runtime -> Restart session, then '
                       'Run all again.')
DEVICE = '0' if cuda_ok else 'cpu'
write_report()
print('\nStep 2 finished. From here on the notebook runs by itself; keep this tab open (or use background execution).')

## Step 3 – Download ReDraw from Zenodo, verify, unpack (1.5–3 h)

* The Zenodo API gives the link, the size (11,579,793,021 bytes) and the MD5 of the archive.
* The download resumes after an interruption (`curl -C -`, up to 40 attempts), and the progress is printed every minute.
* After the MD5 check, the archive is unpacked and deleted.
* The whole step is skipped when training, metrics and the zip are already on Drive.

In [ ]:
#@title Step 3a - Download ReDraw (resumable) and check its MD5
import urllib.request

NEED_DATA = bool(remaining_work())


def zenodo_file_info():
    if DRY_RUN:
        if not DRY.get('record_json'):
            raise PipelineStop('DRY_RUN needs a synthetic archive: set UI2C_DRYRUN_RECORD_JSON; untick DRY_RUN '
                               'for the real run')
        rec, src = json.loads(Path(DRY['record_json']).read_text()), DRY['record_json']
    else:
        src = f'https://zenodo.org/api/records/{ZENODO_RECORD}'
        for attempt in range(1, 6):
            try:
                with urllib.request.urlopen(src, timeout=60) as r:
                    rec = json.loads(r.read().decode('utf-8'))
                break
            except Exception as e:
                print(f'Zenodo API attempt {attempt} failed: {e}')
                if attempt == 5:
                    raise PipelineStop(f'cannot reach the Zenodo API ({src}); try again later')
                time.sleep(20 * attempt)
    f = [x for x in rec.get('files', []) if x.get('key') == ZENODO_FILE]
    if not f:
        raise PipelineStop(f'{ZENODO_FILE} not listed in {src}')
    f = f[0]
    algo, digest = f['checksum'].split(':', 1)
    return {'record': ZENODO_RECORD, 'api': src, 'key': f['key'], 'size': int(f['size']),
            'url': f['links']['self'], 'checksum_algo': algo, 'checksum': digest}


def download_with_resume(url, dest, size, max_attempts=40):
    dest.parent.mkdir(parents=True, exist_ok=True)
    kill_stale(dest, ('curl',))   # a curl of an earlier session (e.g. after a kernel restart) writing the same file
    every = 1 if DRY_RUN else 60
    for attempt in range(1, max_attempts + 1):
        have = dest.stat().st_size if dest.exists() else 0
        if have == size:
            return
        if have > size:
            print(f'{dest} is larger than expected ({have} > {size}); starting again')
            dest.unlink()
            have = 0
        print(f'download attempt {attempt}: {human(have)} of {human(size)} already on disk', flush=True)
        p = subprocess.Popen(['curl', '-L', '-sS', '--fail', '-C', '-', '--connect-timeout', '60',
                              '--speed-limit', '10240', '--speed-time', '180', '-o', str(dest), url],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        t_last, b_last = time.time(), have
        try:
            while True:
                try:
                    p.wait(timeout=every)
                    break
                except subprocess.TimeoutExpired:
                    b = dest.stat().st_size if dest.exists() else 0
                    rate = (b - b_last) / max(time.time() - t_last, 1e-9)
                    t_last, b_last = time.time(), b
                    eta = (size - b) / rate / 60 if rate > 0 else float('inf')
                    print(f'  {human(b)} / {human(size)} ({100 * b / size:.1f} %)  {rate / 1e6:.1f} MB/s  '
                          f'ETA {eta:.0f} min', flush=True)
        except BaseException:   # interrupted: stop curl too, so a later Run all does not start a second one
            p.kill()
            p.wait()
            print('download interrupted; Run all continues where it stopped')
            raise
        msg = p.stdout.read().strip()
        if p.returncode != 0:
            print(f'  curl stopped (exit {p.returncode}): {msg[-300:]}; retrying in {min(120, 15 * attempt)} s')
            time.sleep(0 if DRY_RUN else min(120, 15 * attempt))
    if not dest.exists() or dest.stat().st_size != size:
        raise PipelineStop('the download did not complete after many attempts; Run all again later '
                           '(it resumes where it stopped)')


if not NEED_DATA:
    print('[skip] training, metrics and the zip are already on Drive: the ReDraw data is not needed')
elif done('extract'):
    print(f'[skip] archive already downloaded, verified and unpacked ({marker("extract").get("raw_root")})')
else:
    zinfo = zenodo_file_info()
    tar_path = D['download'] / ZENODO_FILE
    zinfo['local_path'] = str(tar_path)
    save_state('zenodo', zinfo)
    print(f"{zinfo['key']}: {zinfo['size']:,} bytes, {zinfo['checksum_algo']} {zinfo['checksum']}\n  from {zinfo['url']}")
    if not done('download'):
        have = tar_path.stat().st_size if tar_path.exists() else 0
        resumed_from = have
        need = (zinfo['size'] - have) / 1e9 + zinfo['size'] / 1e9 * 1.05 + 2
        if free_gb(WORK) < need:
            raise PipelineStop(f'not enough disk for download + unpacking: {free_gb(WORK):.1f} GB free, '
                               f'~{need:.0f} GB needed')
        t0 = time.time()
        download_with_resume(zinfo['url'], tar_path, zinfo['size'])
        mark('download', {'path': str(tar_path), 'size': zinfo['size'], 'resumed_from_bytes': resumed_from,
                          'minutes': round((time.time() - t0) / 60, 1)})
    if not done('md5'):
        print(f'checking {zinfo["checksum_algo"]} of {tar_path} ...', flush=True)
        got = file_hash(tar_path, zinfo['checksum_algo'], progress_every=0 if DRY_RUN else 30)
        ok = got == zinfo['checksum']
        set_checks('3 download', [check_row(f'{zinfo["checksum_algo"]} of {ZENODO_FILE}', zinfo['checksum'], got),
                                  check_row('size (bytes)', zinfo['size'], tar_path.stat().st_size),
                                  check_row('download resumed from (bytes)', 'reported',
                                            marker('download').get('resumed_from_bytes'), status='INFO')])
        if not ok:
            tar_path.unlink()
            (D['markers'] / 'download.done').unlink(missing_ok=True)
            write_report()
            raise PipelineStop('MD5 mismatch: the downloaded file was deleted. Choose Run all to download it again.')
        mark('md5', {'md5': got})
    else:
        print('[skip] MD5 already verified:', marker('md5').get('md5'))
    write_report()

In [ ]:
#@title Step 3b - Unpack and check the layout
if not NEED_DATA:
    print('[skip] the ReDraw data is not needed')
elif done('extract'):
    xinfo = marker('extract')
    print(f'[skip] already unpacked: {xinfo.get("raw_root")} ({xinfo.get("app_folders"):,} app folders)')
else:
    zinfo = load_state('zenodo')
    tar_path = Path(zinfo['local_path'])
    if not done('md5') or not tar_path.exists():
        raise PipelineStop('the verified archive is missing - run Step 3a first')
    dest = D['raw']
    kill_stale(tar_path, ('tar', 'gzip', 'pigz'))   # an unpack of an earlier session that is still running
    if dest.exists():
        shutil.rmtree(dest)  # a previous unpack was interrupted: start clean
    dest.mkdir(parents=True)
    t0 = time.time()
    print(f'unpacking {tar_path} -> {dest} (5-10 min) ...', flush=True)
    run(['tar', '--no-same-owner', '-xzf', tar_path, '-C', dest], log='extract.log')
    tops = [p for p in dest.iterdir() if p.is_dir()]
    if (dest / 'ReDraw-Final-Google-Play-Dataset').is_dir():
        rr = dest / 'ReDraw-Final-Google-Play-Dataset'
    elif len(tops) == 1:
        rr = tops[0]
    else:
        raise PipelineStop(f'unexpected archive layout in {dest}: {[p.name for p in tops][:10]}')
    apps = [p for p in rr.iterdir() if p.is_dir() and p.name.endswith('-screens')]
    n_xml = sum(1 for a in apps for _ in a.glob('hierarchy_*.xml'))
    n_png = sum(1 for a in apps for _ in a.glob('screenshot_*.png'))
    xinfo = {'raw_root': str(rr), 'app_folders': len(apps), 'hierarchy_xml': n_xml, 'screenshots': n_png,
             'minutes': round((time.time() - t0) / 60, 1)}
    set_checks('3 extraction', [
        check_row('raw root (parse_redraw --raw-root)', 'ReDraw-Final-Google-Play-Dataset', rr.name),
        check_row('app folders <package>-screens', EXPECTED['app_folders_in_archive'], len(apps),
                  status='PASS' if len(apps) == EXPECTED['app_folders_in_archive'] else ('INFO' if DRY_RUN else 'WARN')),
        check_row('hierarchy_<n>.xml files', 'reported', n_xml, status='INFO'),
        check_row('screenshot_<n>.png files', 'reported', n_png, status='INFO')])
    mark('extract', xinfo)
    tar_path.unlink()
    print(f'deleted {tar_path} to free disk space')
    write_report()
print(f'free disk now: {free_gb(WORK):.1f} GB')

## Step 4 – The labelled screens, with the unchanged v1.1.0 code (10–20 min)

`parse_redraw.py` and `build_dataset.py` are run from a clone of tag `v1.1.0`. The commit and the SHA-256 of both scripts and of `configs/dataset.yaml` must equal the pinned values. The commands are those of the v1.1.0 README (with `--symlink`).

The result is the set of **18,694 labelled screens** of the thesis, with the same labels. The v1.1.0 screenshot-level split that `build_dataset` also produces is **not** used: it is only the input of the next step, which re-splits by app.

In [ ]:
#@title Step 4 - The 18,694 labelled screens with the v1.1.0 code (parse_redraw -> build_dataset)
LABELS = D['data'] / f'labels_{REPO_TAG}'          # parse_redraw output: labels/<app>-screens/screenshot_<n>.txt
YOLO_V110 = D['data'] / f'yolo_{REPO_TAG}'         # build_dataset output (screenshot-level split, symlinks)
if not NEED_DATA:
    print('[skip] the ReDraw data is not needed')
else:
    clone, cinfo = clone_repo(REPO_TAG)
    save_state('clone', cinfo)
    code_rows = [check_row(f'{REPO_TAG} commit', PINNED['commit'][REPO_TAG], cinfo['commit'],
                           status=None if not DRY_RUN else ('PASS' if cinfo['commit'] == PINNED['commit'][REPO_TAG] else 'INFO'))]
    code_rows += [check_row(f'{REPO_TAG} sha256 {rel}', PINNED['files'][REPO_TAG][rel][:16] + '...',
                            (h or 'missing')[:16] + '...', ok=h == PINNED['files'][REPO_TAG][rel])
                  for rel, h in cinfo['file_sha256'].items()]
    require(set_checks(f'4 code {REPO_TAG}', code_rows), f'the {REPO_TAG} code check')
    pr = clone / 'src/ui2compose/preprocess/parse_redraw.py'
    bd = clone / 'src/ui2compose/preprocess/build_dataset.py'
    yaml_cfg = clone / 'configs' / 'dataset.yaml'
    rr = raw_root()
    if done('parse'):
        pinfo = load_state(f'parse_{REPO_TAG}')
        print(f'[skip] parse_redraw already done: {pinfo.get("summary")}')
    else:
        kill_stale(LABELS, ('python',))
        if LABELS.exists():
            shutil.rmtree(LABELS)   # start empty: stale label files would change the screen set
        argv = [str(a) for a in ('--raw-root', rr, '--out-root', LABELS, '--dataset-yaml', yaml_cfg)]
        eff = effective_args(pr, argv)
        t0 = time.time()
        print('parse_redraw: reading every XML (10-15 min; no output until it has finished) ...', flush=True)
        _, out = run([sys.executable, pr, *argv], cwd=clone, log='preprocess.log', echo=False)
        m = re.search(r'Total pairs:\s*(\d+),\s*label files written:\s*(\d+)', out)
        pinfo = {'command': qcmd(['python', pr, *argv]), 'effective_args': eff['args'],
                 'summary': m.group(0) if m else None, 'seconds': round(time.time() - t0),
                 'raw_pairs': int(m.group(1)) if m else None, 'label_files_written': int(m.group(2)) if m else None}
        print(pinfo['summary'])
        save_state(f'parse_{REPO_TAG}', pinfo)
        mark('parse', pinfo)
    if done('build'):
        binfo = load_state(f'build_{REPO_TAG}')
        print(f'[skip] build_dataset already done: {binfo.get("summary")}')
    else:
        kill_stale(YOLO_V110, ('python',))
        argv = [str(a) for a in ('--labels-root', LABELS, '--images-root', rr, '--yolo-root', YOLO_V110, '--clean',
                                 '--dataset-yaml', yaml_cfg, '--symlink')]
        eff = effective_args(bd, argv)
        t0 = time.time()
        _, out = run([sys.executable, bd, *argv], cwd=clone, log='preprocess.log')
        m = re.search(r'Total pairs:\s*(\d+)\s*\|\s*train:(\d+)\s+val:(\d+)\s+test:(\d+)', out)
        mu = re.search(r'Unmatched image count[^:]*:\s*(\d+)', out)
        binfo = {'command': qcmd(['python', bd, *argv]), 'effective_args': eff['args'], 'constants': eff['constants'],
                 'summary': m.group(0) if m else None, 'total_pairs': int(m.group(1)) if m else None,
                 'unmatched_images': int(mu.group(1)) if mu else None, 'seconds': round(time.time() - t0)}
        names = read_names(yaml_cfg)
        (YOLO_V110 / 'dataset.yaml').write_text(
            f'path: {YOLO_V110}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(names)}\nnames:\n'
            + ''.join(f'  - {n}\n' for n in names), encoding='utf-8')
        save_state(f'build_{REPO_TAG}', binfo)
        mark('build', binfo)
    NAMES = read_names(yaml_cfg)
    stems = sorted(p.stem for s in SPLITS for p in list_images(YOLO_V110, s))
    n_lab = sum(1 for s in SPLITS for _ in (YOLO_V110 / 'labels' / s).glob('*.txt'))
    a = pinfo.get('effective_args', {})
    E = EXPECTED
    rows = [check_row(f'parse_redraw {k}', v, a.get(k)) for k, v in PINNED['parse_redraw_defaults'].items()]
    rows += [check_row('build_dataset SEED / SPLIT (only used for the v1.1.0 split, not here)', '42 / [0.8, 0.1, 0.1]',
                       f"{binfo['constants'].get('SEED')} / {list(binfo['constants'].get('SPLIT', []))}", status='INFO'),
             check_row('parse_redraw: XML + screenshot pairs', E['raw_pairs'], pinfo.get('raw_pairs'),
                       status=None if not DRY_RUN else 'INFO'),
             check_row('parse_redraw: label files written', E['label_files'], pinfo.get('label_files_written'),
                       status=None if not DRY_RUN else 'INFO'),
             check_row('labelled screens (build_dataset total pairs)', E['total_pairs'], binfo.get('total_pairs'),
                       status=None if not DRY_RUN else 'INFO'),
             check_row('screens in the dataset = total pairs', binfo.get('total_pairs'), len(stems)),
             check_row('every screen has a label file', len(stems), n_lab),
             check_row('stem names unique', len(stems), len(set(stems))),
             check_row('class names (configs/dataset.yaml)', 13, len(NAMES))]
    save_state('screens', {'n': len(stems), 'sha256_all_stems': stems_sha256(stems), 'names': NAMES})
    require(set_checks('4 screens', rows), 'the screen set check')
    write_report()

## Step 5 – App-level split (`app_split --resplit`, v1.2.0), audit, datasets

**Split.** `app_split --resplit --ratios 0.8 0.1 0.1 --seed 42 --near-dup-dist 4` hashes every screen (MD5 + 64-bit dHash).
* A group is a connected component of "same package (case-insensitive) OR identical MD5 OR dHash distance ≤ 4".
* Groups are assigned to train/val/test by greedy multi-label stratification over their class counts, with 80/10/10 of the screens.
* The tool audits its own split. The notebook then audits it **independently**:
  * no group, package, MD5 or identical dHash in two splits;
  * a brute-force search of all dHash pairs ≤ 4 bits;
  * a union-find here that must give exactly the same groups;
  * MD5 and dHash of a random sample recomputed from the image files.
* `build_dataset.py --split-by app` (v1.2.0) must assign every screen to the same split.
* The split fingerprints (SHA-256 of the sorted stems per split) are written to Drive. A later machine must rebuild exactly the same split before it resumes a training run.

**Datasets (Step 5b).**
* `yolo_app_640`: the training copy. The screenshots are resized to 640 px on the long side **exactly as Ultralytics' `load_image` does** (cv2 `INTER_LINEAR`, `min(ceil(w0·r), 640)`) and saved as lossless PNG. Ultralytics then does not resize them again, so the network sees the same pixels as with the 1200×1920 originals, and image I/O is about 9× smaller. A random sample is checked with the real Ultralytics code. The labels are normalised (xywh in 0–1) and are copied unchanged.
* `yolo_app_640_os5k`: the same, plus `oversample_dataset.py` (v1.2.0) with `--target-min 5000` on the **train split only** (8 rare classes: CheckBox, ProgressBar, RadioButton, RecyclerView, SeekBar, Spinner, Switch, WebView). Val and test stay identical, which is checked.
* `yolo_app_full`: the original screenshots (symlinks), used for the final metrics and the zip.

In [ ]:
#@title Step 5 - App-level split with app_split --resplit (v1.2.0), independent audit, fingerprints
import numpy as np

SPLIT_DIR = D['data'] / 'splits' / 'app_level'           # app_split outputs
SPLIT_MANIFEST = SPLIT_DIR / 'app_split_manifest.csv'
DRIVE_SPLIT = DS / 'split'                                 # copy on Drive (a new machine must rebuild the same split)


def popcount64(x):
    if hasattr(np, 'bitwise_count'):
        return np.bitwise_count(x)
    t = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
    return t[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)


def near_dup_pairs_bruteforce(hashes, k):
    """All pairs (i < j) of distinct hash values with Hamming distance <= k, by brute force (independent of the
    multi-index search inside app_split)."""
    h = np.array(hashes, dtype=np.uint64)
    out = []
    for a in range(0, len(h), 256):
        blk = h[a:a + 256]
        d = popcount64(blk[:, None] ^ h[None, :])
        ii, jj = np.nonzero(d <= k)
        for i, j in zip(ii + a, jj):
            if i < j:
                out.append((int(i), int(j), int(d[i - a, j])))
    return out


def own_dhash(path, size=8):
    """dHash re-implemented here (grayscale, Lanczos to 9x8, left < right) to spot-check the manifest."""
    from PIL import Image
    with Image.open(path) as im:
        g = np.asarray(im.convert('L').resize((size + 1, size), Image.Resampling.LANCZOS), dtype=np.int16)
    v = 0
    for b in (g[:, 1:] > g[:, :-1]).ravel():
        v = (v << 1) | int(b)
    return v


if not NEED_DATA:
    print('[skip] the ReDraw data is not needed')
else:
    tool = CODE / 'src/ui2compose/preprocess/app_split.py'
    workers = max(1, os.cpu_count() or 1)
    if done('appsplit') and SPLIT_MANIFEST.exists():
        print(f'[skip] app_split --resplit already done: {SPLIT_MANIFEST}')
    else:
        if SPLIT_DIR.exists():
            shutil.rmtree(SPLIT_DIR)
        args = ['--dataset-dir', YOLO_V110, '--out', SPLIT_DIR, '--dataset-yaml', YOLO_V110 / 'dataset.yaml',
                '--resplit', '--ratios', '0.8', '0.1', '0.1', '--seed', SEED, '--near-dup-dist', NEAR_DUP_DIST,
                '--workers', workers]
        print(f'app_split --resplit: hashing every screen with {workers} processes (A100 machine: ~3-5 min) ...',
              flush=True)
        t0 = time.time()
        run([sys.executable, '-m', 'ui2compose.preprocess.app_split', *args], log='app_split.log')
        mark('appsplit', {'command': qcmd(['python', '-m', 'ui2compose.preprocess.app_split', *args]),
                          'seconds': round(time.time() - t0)})
    with open(SPLIT_MANIFEST, newline='', encoding='utf-8') as f:
        MAN = list(csv.DictReader(f))
    rs_summary = jload(SPLIT_DIR / 'app_split_summary.json')
    stem_split = {r['stem']: r['split'] for r in MAN}
    screens = load_state('screens')
    all_stems = sorted(p.stem for s in SPLITS for p in list_images(YOLO_V110, s))
    path_of = {p.stem: p for s in SPLITS for p in list_images(YOLO_V110, s)}

    # ---- independent audit (computed here, not by app_split) ----
    t0 = time.time()
    where = {'group': defaultdict(set), 'package': defaultdict(set), 'md5': defaultdict(set), 'dhash': defaultdict(set)}
    for r in MAN:
        where['group'][r['group_id']].add(r['split'])
        where['package'][r['package'].lower()].add(r['split'])
        where['md5'][r['md5']].add(r['split'])
        where['dhash'][r['dhash']].add(r['split'])
    viol = {k: sorted(x for x, ss in v.items() if len(ss) > 1) for k, v in where.items()}
    uniq = sorted(where['dhash'])
    splits_of_hash = {h: where['dhash'][h] for h in uniq}
    pairs = near_dup_pairs_bruteforce([int(h, 16) for h in uniq], NEAR_DUP_DIST)
    nd_viol = [(uniq[i], uniq[j], d) for i, j, d in pairs if len(splits_of_hash[uniq[i]] | splits_of_hash[uniq[j]]) > 1]
    # the groups must be exactly the connected components of package OR md5 OR dHash <= k (union-find here)
    idx = {r['stem']: i for i, r in enumerate(MAN)}
    parent = list(range(len(MAN)))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for key in ('package', 'md5', 'dhash'):
        first = {}
        for i, r in enumerate(MAN):
            k = r[key].lower() if key == 'package' else r[key]
            if k in first:
                union(first[k], i)
            else:
                first[k] = i
    first_of_hash = {}
    for i, r in enumerate(MAN):
        first_of_hash.setdefault(r['dhash'], i)
    for i, j, _ in pairs:
        union(first_of_hash[uniq[i]], first_of_hash[uniq[j]])
    comp_of_group = defaultdict(set)
    group_of_comp = defaultdict(set)
    for i, r in enumerate(MAN):
        comp_of_group[r['group_id']].add(find(i))
        group_of_comp[find(i)].add(r['group_id'])
    same_partition = all(len(v) == 1 for v in comp_of_group.values()) and all(len(v) == 1 for v in group_of_comp.values())
    # spot-check the hashes in the manifest against the image files
    rnd = random.Random(SEED)
    sample = rnd.sample(all_stems, min(len(all_stems), 300))
    bad_md5 = [s for s in sample if file_hash(path_of[s], 'md5') != MAN[idx[s]]['md5']]
    bad_dh = [s for s in sample if own_dhash(path_of[s]) != int(MAN[idx[s]]['dhash'], 16)]
    audit_s = round(time.time() - t0)

    # ---- sizes, apps, groups, classes ----
    sizes = Counter(r['split'] for r in MAN)
    n = sum(sizes.values())
    apps = {s: len({r['package'].lower() for r in MAN if r['split'] == s}) for s in SPLITS}
    groups = {s: len({r['group_id'] for r in MAN if r['split'] == s}) for s in SPLITS}
    split_stems = {s: sorted(r['stem'] for r in MAN if r['split'] == s) for s in SPLITS}
    counts = {}
    for s in SPLITS:   # labels of the v1.1.0 dataset, regrouped by the new split
        inst, imgs, nfiles = Counter(), Counter(), 0
        for st in split_stems[s]:
            old = MAN[idx[st]]['old_split']
            here = Counter()
            for ln in (YOLO_V110 / 'labels' / old / f'{st}.txt').read_text(encoding='utf-8').splitlines():
                parts = ln.split()
                if len(parts) >= 5:
                    here[int(float(parts[0]))] += 1
            nfiles += 1
            inst.update(here)
            imgs.update(here.keys())
        counts[s] = {'label_files': nfiles, 'instances': sum(inst.values()),
                     'per_class_instances': {NAMES[c]: inst.get(c, 0) for c in range(len(NAMES))},
                     'per_class_images': {NAMES[c]: imgs.get(c, 0) for c in range(len(NAMES))}}
    fps = {s: stems_sha256(split_stems[s]) for s in SPLITS}
    fp_manifest = hashlib.sha256(''.join(f'{st},{stem_split[st]}\n' for st in sorted(stem_split)).encode()).hexdigest()
    zero_cls = [f'{s}:{c}' for s in ('val', 'test') for c, v in counts[s]['per_class_instances'].items() if v == 0]
    share = {s: sizes[s] / n for s in SPLITS}
    cls_share = {c: {s: counts[s]['per_class_instances'][c] / max(1, sum(counts[t]['per_class_instances'][c]
                                                                         for t in SPLITS)) for s in SPLITS} for c in NAMES}
    worst = max((abs(cls_share[c]['val'] - 0.1), c) for c in NAMES)
    worst_t = max((abs(cls_share[c]['test'] - 0.1), c) for c in NAMES)
    ap_audit = (rs_summary or {}).get('audit', {})

    rows = [check_row('app_split audit (inside app_split --resplit)', True, bool(ap_audit.get('ok')),
                      note=f"checks {ap_audit.get('checks')}"),
            check_row('every screen of Step 4 is in the manifest exactly once', len(all_stems),
                      len(stem_split), ok=sorted(stem_split) == all_stems and len(MAN) == len(all_stems)),
            check_row('groups in two splits (independent check)', 0, len(viol['group']), note=', '.join(viol['group'][:3])),
            check_row('packages in two splits (case-insensitive)', 0, len(viol['package']),
                      note=', '.join(viol['package'][:3])),
            check_row('identical MD5 in two splits', 0, len(viol['md5'])),
            check_row('identical dHash in two splits', 0, len(viol['dhash'])),
            check_row(f'dHash pairs <= {NEAR_DUP_DIST} bits in two splits (brute force)', 0, len(nd_viol),
                      note=f'{len(pairs):,} near-duplicate pairs of distinct hashes in total'),
            check_row('groups = connected components of package/MD5/dHash (union-find here)', True, same_partition,
                      note=f"{len(comp_of_group):,} groups"),
            check_row('manifest MD5 = MD5 of the image file (random sample)', f'0 of {len(sample)} differ', f'{len(bad_md5)} differ',
                      ok=not bad_md5),
            check_row('manifest dHash = dHash recomputed here (random sample)', f'0 of {len(sample)} differ',
                      f'{len(bad_dh)} differ', ok=not bad_dh),
            check_row('split parameters (ratios, seed, near-dup k, builder prefix)', f'[0.8, 0.1, 0.1], {SEED}, {NEAR_DUP_DIST}, False',
                      f"{rs_summary['parameters']['ratios']}, {rs_summary['parameters']['seed']}, "
                      f"{rs_summary['parameters']['near_dup_max_dist']}, {rs_summary['parameters']['builder_prefix']}"),
            check_row('screens train/val/test', '~80 / 10 / 10 %', f"{sizes['train']:,} / {sizes['val']:,} / {sizes['test']:,}",
                      status='PASS' if all(abs(share[s] - r) <= 0.015 for s, r in zip(SPLITS, (0.8, 0.1, 0.1)))
                      else ('INFO' if DRY_RUN else 'WARN'),
                      note=' / '.join(f'{100 * share[s]:.1f} %' for s in SPLITS)),
            check_row('apps train/val/test', 'reported', f"{apps['train']:,} / {apps['val']:,} / {apps['test']:,}", status='INFO'),
            check_row('groups train/val/test', 'reported', f"{groups['train']:,} / {groups['val']:,} / {groups['test']:,}",
                      status='INFO', note=f"largest group {max(Counter(r['group_id'] for r in MAN).values()):,} screens"),
            check_row('classes without instances in val or test', 0, len(zero_cls),
                      status='PASS' if not zero_cls else ('INFO' if DRY_RUN else 'WARN'), note=', '.join(zero_cls)),
            check_row('largest deviation of a class share from 10 % (val / test)', 'reported',
                      f'{100 * worst[0]:.1f} pt ({worst[1]}) / {100 * worst_t[0]:.1f} pt ({worst_t[1]})', status='INFO'),
            check_row('audit run time (s)', 'reported', audit_s, status='INFO')]
    for s in SPLITS:
        rows.append(check_row(f'{s} fingerprint (sha256 of sorted stems)', 'recorded', fps[s][:16] + '...', status='INFO'))

    # ---- the split must be the one already used on Drive (a new machine rebuilds it from scratch) ----
    prev = jload(DRIVE_SPLIT / 'split_fingerprints.json')
    cur = {'splits': fps, 'manifest_sha256': fp_manifest, 'sizes': dict(sizes), 'seed': SEED,
           'near_dup_dist': NEAR_DUP_DIST, 'created': now()}
    if prev:
        same = prev['splits'] == fps and prev['manifest_sha256'] == fp_manifest
        rows.append(check_row('split = the split already on Drive (earlier machine)', prev['manifest_sha256'][:16] + '...',
                              fp_manifest[:16] + '...', ok=same, note=f"recorded {prev.get('created')}"))
    else:
        rows.append(check_row('split recorded on Drive for later machines', 'first run', 'written', status='INFO'))
    split_ok = set_checks('5 app-level split', rows)
    save_state('split', {'sizes': dict(sizes), 'apps': apps, 'groups': groups, 'seed': SEED, 'counts': counts,
                         'instances': {s: counts[s]['instances'] for s in SPLITS}, 'fingerprints': fps,
                         'manifest_sha256': fp_manifest, 'audit_ok': split_ok, 'near_dup_pairs': len(pairs),
                         'class_share': cls_share})
    require(split_ok, 'the app-level split audit')
    if not prev:
        DRIVE_SPLIT.mkdir(parents=True, exist_ok=True)
        jdump(cur, DRIVE_SPLIT / 'split_fingerprints.json')
    for p in (SPLIT_MANIFEST, SPLIT_DIR / 'app_split_summary.json'):
        copy_to_drive(p, DRIVE_SPLIT / p.name)
    for s in SPLITS:
        (SPLIT_DIR / f'split_{s}_stems.txt').write_text(''.join(x + '\n' for x in split_stems[s]), encoding='utf-8')

    # ---- cross-check: build_dataset.py --split-by app (v1.2.0) must give the same split ----
    if CROSS_CHECK_BUILD_DATASET:
        xdir = D['data'] / 'yolo_crosscheck_build_dataset_app'
        if not done('xcheck'):
            bd2 = CODE / 'src/ui2compose/preprocess/build_dataset.py'
            argv = ['--labels-root', LABELS, '--images-root', raw_root(), '--yolo-root', xdir, '--clean', '--symlink',
                    '--dataset-yaml', YOLO_V110 / 'dataset.yaml', '--split-by', 'app', '--near-dup-dist', NEAR_DUP_DIST,
                    '--workers', workers]
            print('\ncross-check: build_dataset.py --split-by app (hashes every screen again) ...', flush=True)
            run([sys.executable, '-m', 'ui2compose.preprocess.build_dataset', *argv], log='crosscheck.log')
            shutil.copyfile(xdir / 'split_manifest.csv', SPLIT_DIR / 'build_dataset_split_manifest.csv')
            shutil.rmtree(xdir)   # only symlinks and labels
            mark('xcheck', {'command': qcmd(['python', '-m', 'ui2compose.preprocess.build_dataset', *argv])})
        with open(SPLIT_DIR / 'build_dataset_split_manifest.csv', newline='', encoding='utf-8') as f:
            xs = {r['stem']: r['split'] for r in csv.DictReader(f)}
        diff = sorted(s for s in set(xs) | set(stem_split) if xs.get(s) != stem_split.get(s))
        ok = set_checks('5 cross-check build_dataset --split-by app', [
            check_row('screens in both manifests', len(stem_split), len(xs)),
            check_row('screens assigned to a different split', 0, len(diff), note=', '.join(diff[:3]))])
        require(ok, 'the build_dataset --split-by app cross-check')
    print('\n' + '\n'.join(class_table(counts)))
    write_report()

In [ ]:
#@title Step 5b - YOLO datasets of the new split: pre-resized (training), full size (evaluation), oversampled (os5k)
import yaml as _yaml
from concurrent.futures import ProcessPoolExecutor

CFG_SRC = {r: CODE / 'configs' / f'train_{r}.yaml' for r in RUNS}
TRAIN_CFG = {r: _yaml.safe_load(CFG_SRC[r].read_text(encoding='utf-8')) for r in RUNS}
TRAIN_IMGSZ = DRY['imgsz'] if DRY_RUN else int(TRAIN_CFG[RUNS[0]]['train']['imgsz'])
YOLO_640 = D['data'] / f'yolo_app_{TRAIN_IMGSZ}'            # training (baseline): long side pre-resized to imgsz
YOLO_OS5K = D['data'] / f'yolo_app_{TRAIN_IMGSZ}_os5k'      # training (oversample_5k): + oversampled copies
YOLO_FULL = D['data'] / 'yolo_app_full'                     # evaluation + zip: original screenshots (symlinks)
DATA_YAML = {'baseline': YOLO_640 / 'dataset.yaml', 'oversample_5k': YOLO_OS5K / 'dataset.yaml'}

_RESIZE_WORKER = r'''
import math, sys, cv2, numpy as np
from ultralytics.utils.patches import imread
def resize_one(args):
    """Exactly what ultralytics BaseDataset.load_image does (rect mode) before augmentation: imread (BGR),
    long side to imgsz with cv2.INTER_LINEAR, size min(ceil(w0 * r), imgsz). Written as lossless PNG and read
    back to prove the file decodes to the same array."""
    src, dst, imgsz = args
    im = imread(src, flags=cv2.IMREAD_COLOR)
    h0, w0 = im.shape[:2]
    r = imgsz / max(h0, w0)
    if r != 1:
        w, h = (min(math.ceil(w0 * r), imgsz), min(math.ceil(h0 * r), imgsz))
        im = cv2.resize(im, (w, h), interpolation=cv2.INTER_LINEAR)
    if not cv2.imwrite(dst, im):
        raise IOError(f'cannot write {dst}')
    back = imread(dst, flags=cv2.IMREAD_COLOR)
    return h0, w0, im.shape[0], im.shape[1], bool(back is not None and np.array_equal(back, im))
'''


def write_yaml(root, names):
    (Path(root) / 'dataset.yaml').write_text(
        f'path: {Path(root).resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(names)}\n'
        'names:\n' + ''.join(f'  - {n}\n' for n in names), encoding='utf-8')


def ultralytics_load(path, imgsz):
    """Runs the real ultralytics BaseDataset.load_image on one file (no dataset object needed)."""
    import cv2
    from types import SimpleNamespace
    from ultralytics.data.base import BaseDataset
    ns = SimpleNamespace(ims=[None], im_files=[str(path)], npy_files=[Path('/nonexistent/none.npy')], imgsz=imgsz,
                         cv2_flag=cv2.IMREAD_COLOR, augment=False)
    return BaseDataset.load_image(ns, 0, rect_mode=True)


if not NEED_DATA:
    print('[skip] the ReDraw data is not needed')
else:
    src_of = {r['stem']: YOLO_V110 / 'images' / r['old_split'] / f"{r['stem']}{path_of[r['stem']].suffix}" for r in MAN}
    lab_of = {r['stem']: YOLO_V110 / 'labels' / r['old_split'] / f"{r['stem']}.txt" for r in MAN}
    # ---- full-size dataset (symlinks to the ReDraw screenshots), used for evaluation and the zip ----
    if not done('ds_full'):
        if YOLO_FULL.exists():
            shutil.rmtree(YOLO_FULL)
        for s in SPLITS:
            (YOLO_FULL / 'images' / s).mkdir(parents=True)
            (YOLO_FULL / 'labels' / s).mkdir(parents=True)
        for r in MAN:
            st, s = r['stem'], r['split']
            (YOLO_FULL / 'images' / s / src_of[st].name).symlink_to(src_of[st].resolve())
            shutil.copyfile(lab_of[st], YOLO_FULL / 'labels' / s / f'{st}.txt')
        write_yaml(YOLO_FULL, NAMES)
        mark('ds_full', {'root': str(YOLO_FULL)})
    # ---- pre-resized dataset for training ----
    if not done('ds_resized'):
        if YOLO_640.exists():
            shutil.rmtree(YOLO_640)
        for s in SPLITS:
            (YOLO_640 / 'images' / s).mkdir(parents=True)
            (YOLO_640 / 'labels' / s).mkdir(parents=True)
        wfile = D['tools'] / 'resize_worker.py'
        wfile.write_text(_RESIZE_WORKER, encoding='utf-8')
        sys.path.insert(0, str(D['tools']))
        import importlib
        resize_worker = importlib.import_module('resize_worker')
        jobs = [(str(src_of[r['stem']].resolve()), str(YOLO_640 / 'images' / r['split'] / f"{r['stem']}.png"),
                 TRAIN_IMGSZ) for r in MAN]
        t0 = time.time()
        nproc = max(1, (os.cpu_count() or 2))
        print(f'pre-resizing {len(jobs):,} screenshots to {TRAIN_IMGSZ} px (long side) with {nproc} processes ...',
              flush=True)
        if nproc > 1 and len(jobs) > 50:
            with ProcessPoolExecutor(max_workers=nproc) as ex:
                res = list(ex.map(resize_worker.resize_one, jobs, chunksize=64))
        else:
            res = [resize_worker.resize_one(j) for j in jobs]
        for r in MAN:
            shutil.copyfile(lab_of[r['stem']], YOLO_640 / 'labels' / r['split'] / f"{r['stem']}.txt")
        write_yaml(YOLO_640, NAMES)
        sizes0 = Counter(f'{w0}x{h0}' for h0, w0, h, w, ok in res)
        sizes1 = Counter(f'{w}x{h}' for h0, w0, h, w, ok in res)
        info = {'root': str(YOLO_640), 'imgsz': TRAIN_IMGSZ, 'images': len(res), 'readback_ok': sum(ok for *_, ok in res),
                'original_sizes': dict(sizes0.most_common(8)), 'resized_sizes': dict(sizes1.most_common(8)),
                'seconds': round(time.time() - t0),
                'method': 'ultralytics imread (IMREAD_COLOR) + cv2.resize INTER_LINEAR to long side imgsz, '
                          'size min(ceil(w0*r), imgsz) = ultralytics BaseDataset.load_image; lossless PNG; '
                          'labels copied unchanged (normalised xywh)'}
        with open(D['data'] / 'resize_sizes.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            w.writerow(['stem', 'split', 'w0', 'h0', 'w', 'h'])
            for r, (h0, w0, h, ww, ok) in zip(MAN, res):
                w.writerow([r['stem'], r['split'], w0, h0, ww, h])
        mark('ds_resized', info)
    info = marker('ds_resized')
    # identity check with the real ultralytics code: original file -> load_image == pre-resized file -> load_image
    import numpy as np
    rnd = random.Random(SEED + 1)
    chk = rnd.sample([r['stem'] for r in MAN], min(len(MAN), 200))
    split_of = {r['stem']: r['split'] for r in MAN}
    mism = []
    for st in chk:
        a, _, hwa = ultralytics_load(src_of[st].resolve(), TRAIN_IMGSZ)
        b, _, hwb = ultralytics_load(YOLO_640 / 'images' / split_of[st] / f'{st}.png', TRAIN_IMGSZ)
        if hwa != hwb or not np.array_equal(a, b):
            mism.append(st)
    n_files = {s: len(list_images(YOLO_640, s)) for s in SPLITS}
    rows = [check_row('pre-resized images written and read back identical', len(MAN), info['readback_ok']),
            check_row('images per split (pre-resized) = manifest', f"{sizes['train']:,}/{sizes['val']:,}/{sizes['test']:,}",
                      f"{n_files['train']:,}/{n_files['val']:,}/{n_files['test']:,}",
                      ok=all(n_files[s] == sizes[s] for s in SPLITS)),
            check_row('ultralytics load_image(original) == load_image(pre-resized) (random sample)',
                      f'0 of {len(chk)} differ', f'{len(mism)} differ', ok=not mism, note=', '.join(mism[:3])),
            check_row('original screenshot sizes', 'reported', json.dumps(info['original_sizes']), status='INFO'),
            check_row('pre-resized sizes', 'reported', json.dumps(info['resized_sizes']), status='INFO',
                      note='labels are normalised, copied unchanged'),
            check_row('full-size evaluation dataset (symlinks)', len(MAN),
                      sum(len(list_images(YOLO_FULL, s)) for s in SPLITS))]
    ok = set_checks('5b datasets', rows)
    require(ok, 'the dataset check')

    # ---- oversample_5k: v1.2.0 oversample_dataset.py on the TRAIN split only ----
    if 'oversample_5k' in RUNS:
        if not done('ds_os5k'):
            if YOLO_OS5K.exists():
                shutil.rmtree(YOLO_OS5K)
            for sub in ('images', 'labels'):
                for s in SPLITS:
                    (YOLO_OS5K / sub / s).mkdir(parents=True)
                    for p in sorted((YOLO_640 / sub / s).iterdir()):
                        os.link(p, YOLO_OS5K / sub / s / p.name)   # hard links: no extra disk
            write_yaml(YOLO_OS5K, NAMES)
            osp = CODE / 'src/ui2compose/preprocess/oversample_dataset.py'
            argv = ['--yolo-root', YOLO_OS5K, '--target-min', OVERSAMPLE_TARGET, '--seed', SEED]
            t0 = time.time()
            run([sys.executable, osp, *argv], log='oversample.log')
            mark('ds_os5k', {'command': qcmd(['python', osp, *argv]), 'seconds': round(time.time() - t0)})
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location('ui2c_os', CODE / 'src/ui2compose/preprocess/oversample_dataset.py')
        _osm = _ilu.module_from_spec(_spec)
        _spec.loader.exec_module(_osm)
        rare = sorted(_osm.CLASS_NAMES[c] for c in _osm.RARE_CLASSES)
        base_tr = count_labels(YOLO_640, NAMES, 'train')
        os_tr = count_labels(YOLO_OS5K, NAMES, 'train')
        os_imgs = {s: [p.stem for p in list_images(YOLO_OS5K, s)] for s in SPLITS}
        copies = [x for x in os_imgs['train'] if re.search(r'_os\d+$', x)]
        originals = sorted(x for x in os_imgs['train'] if not re.search(r'_os\d+$', x))
        bad_copy = [x for x in copies if re.sub(r'(_os\d+)+$', '', x) not in set(originals)]
        cs = rnd.sample(copies, min(len(copies), 100))
        not_ident = [x for x in cs if file_hash(YOLO_OS5K / 'images/train' / f'{x}.png', 'md5') != file_hash(
            YOLO_640 / 'images/train' / f"{re.sub(r'(_os[0-9]+)+$', '', x)}.png", 'md5')]
        rows = [check_row('rare classes of oversample_dataset.py', ', '.join(RARE_CLASS_NAMES), ', '.join(rare)),
                check_row('target instances per rare class', 5000, OVERSAMPLE_TARGET),
                check_row('only the train split was oversampled: val stems unchanged', fps['val'][:16] + '...',
                          stems_sha256(os_imgs['val'])[:16] + '...', ok=stems_sha256(os_imgs['val']) == fps['val']),
                check_row('only the train split was oversampled: test stems unchanged', fps['test'][:16] + '...',
                          stems_sha256(os_imgs['test'])[:16] + '...', ok=stems_sha256(os_imgs['test']) == fps['test']),
                check_row('original train screens unchanged', fps['train'][:16] + '...', stems_sha256(originals)[:16] + '...',
                          ok=stems_sha256(originals) == fps['train']),
                check_row('every copy is named <train stem>_os<k>', 0, len(bad_copy)),
                check_row('copies byte-identical to their source (random sample)', f'0 of {len(cs)} differ',
                          f'{len(not_ident)} differ', ok=not not_ident),
                check_row('train images: originals + copies', f'{len(originals):,} + copies',
                          f'{len(originals):,} + {len(copies):,} = {len(os_imgs["train"]):,}', status='INFO')]
        for c in RARE_CLASS_NAMES:
            rows.append(check_row(f'{c} train instances before -> after', 'reported',
                                  f"{base_tr['per_class_instances'][c]:,} -> {os_tr['per_class_instances'][c]:,}",
                                  status='INFO', note=f"images {base_tr['per_class_images'][c]:,} -> "
                                                      f"{os_tr['per_class_images'][c]:,}"))
        ok = set_checks('5b oversampling', rows)
        ds = load_state('datasets') or {}
        ds['os5k'] = {'train_images': len(os_imgs['train']), 'copies': len(copies), 'originals': len(originals),
                      'rare_after': {c: os_tr['per_class_instances'][c] for c in RARE_CLASS_NAMES},
                      'per_class_before': base_tr, 'per_class_after': os_tr}
        save_state('datasets', ds)
        print('\n' + '\n'.join(class_table({'baseline': base_tr, 'os5k': os_tr}, splits=('baseline', 'os5k'))))
        require(ok, 'the oversampling check')
    write_report()

## Step 6 – Training (baseline, then oversample_5k) and metrics of each run

* `train.py` (v1.2.0) runs with `configs/train_baseline.yaml` and `configs/train_oversample_5k.yaml`. Only the paths (`model`, `data`, `project`, `name`) and the cache mode change. All hyperparameters are checked twice against the released checkpoint's `train_args`: before training (resolved by `train.py --dry-run`) and after it (`args.yaml`).
* Initial weights: COCO-pretrained `yolov8s.pt` (Ultralytics asset; SHA-256 recorded).
* `cache`: `ram` (as in the released run) if the decoded images fit into RAM, otherwise `disk`. Disk caching uses the same decoded arrays and is only slower.
* **Checkpoints on Drive:** a launcher runs `train.py` unchanged. It only registers an Ultralytics callback. After every epoch, that callback copies `last.pt`, and `best.pt` when it improved, to `runs/<run>_app/` on Drive, with SHA-256, the epoch time and an ETA (`progress.json`). `save_period` stays -1 as in the released run; it would only add one extra 67 MB `epoch<N>.pt` file per epoch.
* **Resume:** a crashed training restarts automatically from `last.pt`, up to 3 times. On a new machine, *Run all* restores the newest checkpoint from Drive and resumes with `train.py --resume`.
* **Time:** after the first epoch the notebook prints the time per epoch, the projected time of this run and of the next run.
* **Metrics after each run:** `YOLO.val` on the NEW val and test splits, with the original screenshots, conf 0.001, NMS IoU 0.7, imgsz 640, max_det 300, batch 16 and rect. FP32 is always run; FP16 is added on a GPU, as the training-time validation does. JSONs go to `metrics/` on Drive.

In [ ]:
#@title Step 6a - Training setup: initial weights, cache mode, configs, hyperparameter check, Drive checkpoint hook
import yaml as _yaml

INIT = D['weights'] / INIT_WEIGHTS
REF_TRAIN_ARGS = jload(CODE / 'results' / 'train_args_oversample_5k.json')
LAUNCHER = D['tools'] / 'train_with_drive_checkpoints.py'

# train.py runs unchanged (its main() resolves the config and calls ultralytics.YOLO(model).train(**kwargs)).
# The launcher only replaces ultralytics.YOLO by a subclass that registers one callback:
#   on_model_save    -> copy last.pt (and best.pt when it improved) to Drive, alternating between two slots, plus
#                       results.csv and an index with SHA-256 (a copy interrupted by a lost machine never
#                       replaces the last good one);
#   it also writes the epoch time, the ETA and the validation metrics to Drive (progress.json) and prints them.
_LAUNCHER = r'''
import hashlib, json, os, shutil, sys, time
from pathlib import Path
H = json.loads(os.environ["UI2C_HOOK"])
DR = Path(H["drive_run"]); DR.mkdir(parents=True, exist_ok=True)
import ultralytics
_Base = ultralytics.YOLO
SEEN = set()

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(8 << 20), b""):
            h.update(c)
    return h.hexdigest()

def jl(p, d):
    try:
        return json.loads(Path(p).read_text())
    except Exception:
        return d

def drive_free():
    try:
        return shutil.disk_usage(H["drive_mount"]).free / 1e9
    except OSError:
        return None

def put(src, name):
    dest = DR / name
    shutil.copyfile(src, dest)   # overwrite in place (no deleted files in the Drive bin)
    h = sha(src)
    return h if sha(dest) == h else None

def on_model_save(tr):
    ep = tr.epoch + 1
    final = bool(tr.stop) or ep >= tr.epochs
    report_epoch(tr, time.time() - tr.epoch_time_start)   # epoch time incl. validation, recorded before the copy
    idx = jl(DR / "ckpt_index.json", {})
    free = drive_free()
    due = ep % max(1, H["every"]) == 0 or final or ep == 1
    if due and (free is None or free >= H["reserve_gb"]):
        try:
            t0 = time.time()
            slot = f"last_slot{ep % 2}.pt"
            h = put(tr.last, slot)
            if h:
                idx["last"] = {"epoch": ep, "file": slot, "sha256": h, "stop": final, "epochs": tr.epochs,
                               "time": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())}
            if tr.best.exists() and tr.best_fitness == tr.fitness:
                bslot = f"best_slot{idx.get('best_next', 0)}.pt"
                hb = put(tr.best, bslot)
                if hb:
                    idx["best"] = {"epoch": ep, "file": bslot, "sha256": hb, "fitness": float(tr.best_fitness)}
                    idx["best_next"] = 1 - idx.get("best_next", 0)
            shutil.copyfile(tr.csv, DR / "results.csv")
            idx["copies"] = idx.get("copies", 0) + 1
            idx["bytes_copied"] = idx.get("bytes_copied", 0) + Path(tr.last).stat().st_size
            (DR / "ckpt_index.json").write_text(json.dumps(idx, indent=1))
            print(f"[ui2compose] epoch {ep}: last.pt copied to Drive ({slot}, {time.time() - t0:.0f} s; "
                  f"Drive free {free if free is None else round(free, 1)} GB)", flush=True)
        except Exception as e:   # Drive trouble must never stop the training
            print(f"[ui2compose] WARNING: checkpoint copy to Drive failed at epoch {ep}: {e}", flush=True)
    elif due:
        print(f"[ui2compose] WARNING: Drive has only {free:.2f} GB free (< {H['reserve_gb']} GB kept for the zip): "
              f"checkpoint of epoch {ep} NOT copied", flush=True)
    c = H.get("crash")
    if c and ep == c["after_epoch"] and not Path(c["flag"]).exists():   # DRY_RUN only: simulated crash
        Path(c["flag"]).write_text(f"crashed after epoch {ep}")
        print(f"[ui2compose] DRY_RUN: simulating a crash after epoch {ep} (exit {c['code']})", flush=True)
        os._exit(c["code"])

def report_epoch(tr, et):
    if tr.epoch in SEEN:
        return
    SEEN.add(tr.epoch)
    ep = tr.epoch + 1
    prog = jl(DR / "progress.json", {"epoch_times_s": {}})
    prog["epoch_times_s"][str(ep)] = round(et, 1)
    times = list(prog["epoch_times_s"].values())
    mean = sum(times) / len(times)
    left = max(0, tr.epochs - ep)
    eta = time.time() + left * mean
    prog.update({"run": H["run"], "epoch": ep, "epochs": tr.epochs, "last_epoch_s": round(et, 1),
                 "mean_epoch_s": round(mean, 1), "remaining_h": round(left * mean / 3600, 2),
                 "eta_utc": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime(eta)), "gpu": H.get("gpu"),
                 "fitness": float(tr.fitness or 0), "best_fitness": float(tr.best_fitness or 0),
                 "metrics": {k: float(v) for k, v in (tr.metrics or {}).items()},
                 "updated": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())})
    try:
        (DR / "progress.json").write_text(json.dumps(prog, indent=1))
    except Exception as e:
        print(f"[ui2compose] WARNING: progress.json not written: {e}", flush=True)
    m = tr.metrics or {}
    line = (f"[ui2compose] {H['run']} epoch {ep}/{tr.epochs}: {et / 60:.1f} min; val mAP50 "
            f"{m.get('metrics/mAP50(B)', float('nan')):.4f} mAP50-95 {m.get('metrics/mAP50-95(B)', float('nan')):.4f}; "
            f"remaining {left} epochs ~ {left * mean / 3600:.1f} h (ETA {prog['eta_utc']})")
    if len(SEEN) == 1:
        bar = "=" * 100
        print(f"{bar}\n[ui2compose] TIME ESTIMATE after the first epoch of this session: {et / 60:.1f} min per "
              f"epoch -> {left * et / 3600:.1f} h for the remaining {left} epochs of {H['run']} "
              f"(ETA {prog['eta_utc']}); with early stopping it can be less.", flush=True)
        n = H.get("n_train", {})
        for other in H.get("later_runs", []):   # scaled by the number of training images per epoch
            if n.get(other) and n.get(H["run"]):
                est = et * n[other] / n[H["run"]] * H.get("epochs_of", {}).get(other, tr.epochs) / 3600
                print(f"[ui2compose] projected {other}: ~{est:.1f} h ({n[other]:,} vs {n[H['run']]:,} training "
                      f"images per epoch)", flush=True)
        print(bar, flush=True)
    print(line, flush=True)

class YOLO(_Base):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.add_callback("on_model_save", on_model_save)

ultralytics.YOLO = YOLO
from ui2compose import train
sys.exit(train.main(sys.argv[1:]))
'''

if not training_left():
    print('[skip] every training run is finished (Drive markers)')
else:
    LAUNCHER.write_text(_LAUNCHER, encoding='utf-8')
    # ---- initial weights: COCO-pretrained yolov8s.pt ----
    if not INIT.exists():
        if DRY_RUN and DRY.get('yolov8s'):
            shutil.copyfile(DRY['yolov8s'], INIT)
        else:
            from ultralytics.utils.downloads import attempt_download_asset
            attempt_download_asset(str(INIT))
    init_sha = file_hash(INIT)
    # ---- cache mode: RAM if the decoded training + validation images fit (Ultralytics' own 50 % margin + 6 GB) ----
    import psutil
    rs = Counter()
    with open(D['data'] / 'resize_sizes.csv', newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            rs[r['split']] += int(r['w']) * int(r['h']) * 3
    n_train = {'baseline': sizes['train'], 'oversample_5k': (load_state('datasets') or {}).get('os5k', {}).get('train_images', 0)}
    mean_train = rs['train'] / max(1, sizes['train'])
    CACHE = {}
    for r in RUNS:
        need_ram = (n_train[r] * mean_train + rs['val']) * 1.5 / 1e9
        avail = psutil.virtual_memory().available / 1e9 - 6
        if CACHE_MODE in ('ram', 'disk'):
            CACHE[r] = CACHE_MODE
        else:
            CACHE[r] = 'ram' if need_ram <= avail else 'disk'
        print(f'{r}: {n_train[r]:,} training images; decoded images need ~{need_ram:.1f} GB RAM, '
              f'{avail:.1f} GB available -> cache={CACHE[r]}')
    # ---- derived configs: the repository's configs/train_<run>.yaml with only paths and cache changed ----
    CFG_RUN, RESOLVED = {}, {}
    for r in RUNS:
        cfg = _yaml.safe_load(CFG_SRC[r].read_text(encoding='utf-8'))
        changed = {'model': (cfg['model'], str(INIT)), 'data': (cfg['data'], str(DATA_YAML[r])),
                   'project': (cfg.get('project'), str(D['runs'])), 'name': (cfg.get('name'), f'{r}_app'),
                   'train.cache': (cfg['train'].get('cache'), CACHE[r])}
        cfg.update(model=str(INIT), data=str(DATA_YAML[r]), project=str(D['runs']), name=f'{r}_app')
        cfg['train']['cache'] = CACHE[r]
        if DRY_RUN:   # tiny CPU run
            for k, v in (('epochs', int(DRY['epochs'].get(r, 1))), ('imgsz', DRY['imgsz']), ('workers', DRY['workers']),
                         ('plots', False)):
                changed[f'train.{k}'] = (cfg['train'].get(k), v)
                cfg['train'][k] = v
        p = D['cfg'] / f'train_{r}_app.yaml'
        p.write_text(f'# {CFG_SRC[r].name} of ui2compose v1.2.0 with these keys changed: '
                     + json.dumps({k: v[1] for k, v in changed.items()}) + '\n'
                     + _yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
        CFG_RUN[r] = p
        _, out = run([sys.executable, '-m', 'ui2compose.train', '--config', p, '--device', DEVICE, '--dry-run'],
                     echo=False, quiet=True, log='train_resolve.log')
        RESOLVED[r] = json.loads(out[out.index('{'):])
        kw = RESOLVED[r]['train_kwargs']
        rows = []
        for k, v in EXPECTED_TRAIN_ARGS.items():
            dry_override = DRY_RUN and f'train.{k}' in changed
            rows.append(check_row(f'{k}', v, kw.get(k), status='INFO' if dry_override else None,
                                  note='DRY_RUN override' if dry_override else ''))
        rows.append(check_row('initial weights (COCO-pretrained YOLOv8s)', 'yolov8s.pt', Path(RESOLVED[r]['model']).name))
        rows.append(check_row('initial weights sha256', INIT_WEIGHTS_SHA256[:16] + '...', init_sha[:16] + '...',
                              status='PASS' if init_sha == INIT_WEIGHTS_SHA256 else 'WARN',
                              note='same file as used for the thesis audit' if init_sha == INIT_WEIGHTS_SHA256 else
                              'a different upload of the Ultralytics yolov8s.pt asset'))
        diff = sorted(k for k in kw if k in REF_TRAIN_ARGS and k not in TRAIN_ARGS_MAY_DIFFER
                      and REF_TRAIN_ARGS[k] != kw[k] and not (DRY_RUN and f'train.{k}' in changed))
        rows.append(check_row('every other key = released checkpoint train_args', 0, len(diff),
                              note=', '.join(f'{k}: {REF_TRAIN_ARGS[k]} -> {kw[k]}' for k in diff[:6])))
        rows.append(check_row('cache', 'ram (checkpoint: cache=True)', kw.get('cache'),
                              status='PASS' if kw.get('cache') == 'ram' else 'WARN',
                              note='disk: not enough RAM; same decoded images, only slower' if kw.get('cache') != 'ram' else ''))
        rows.append(check_row('ultralytics version in the config', ULTRALYTICS_VERSION, RESOLVED[r]['ultralytics_version']))
        rows.append(check_row('changed keys (paths, cache' + (', DRY_RUN' if DRY_RUN else '') + ')', 'reported',
                              json.dumps({k: v[1] for k, v in changed.items()}), status='INFO'))
        require(set_checks(f'6 {r} hyperparameters', rows), f'the hyperparameter check of {r}')
    N_TRAIN = n_train
    save_state('train_setup', {'init_sha256': init_sha, 'cache': CACHE, 'configs': {r: str(p) for r, p in CFG_RUN.items()},
                               'resolved': RESOLVED})
    write_report()

In [ ]:
#@title Step 6b - Train baseline, then oversample_5k (resumes from Drive), evaluate each run on the new val + test
import torch


def ckpt_epoch(p):
    """Completed epochs of a checkpoint (Ultralytics stores the 0-based epoch; -1 = finished and stripped).
    None if the file cannot be read."""
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
        return int(ck.get('epoch', -1))
    except Exception as e:
        print(f'  cannot read {p}: {type(e).__name__}: {e}')
        return None


def truncate_results(csv_path, n_epochs):
    """Keeps the header and the rows of epochs 1..n_epochs (a row can be ahead of last.pt after a crash)."""
    if not Path(csv_path).exists():
        return
    lines = Path(csv_path).read_text(encoding='utf-8').splitlines()
    keep = [lines[0]] + [l for l in lines[1:] if l.strip() and float(l.split(',')[0]) <= n_epochs]
    Path(csv_path).write_text('\n'.join(keep) + '\n', encoding='utf-8')


def restore_from_drive(run_name, local):
    """Copies the newest good checkpoint (SHA-256 checked, else the other slot) from Drive into local/weights."""
    dr = DRUNS / f'{run_name}_app'
    idx = jload(dr / 'ckpt_index.json', {})
    if not idx.get('last'):
        return None
    w = local / 'weights'
    w.mkdir(parents=True, exist_ok=True)
    cands = [idx['last']['file']] + [f for f in ('last_slot0.pt', 'last_slot1.pt') if f != idx['last']['file']]
    got = None
    for f in cands:
        if not (dr / f).exists():
            continue
        shutil.copyfile(dr / f, w / 'last.pt')
        if f == idx['last']['file'] and file_hash(w / 'last.pt') == idx['last']['sha256']:
            got = (f, idx['last']['epoch'])
            break
        e = ckpt_epoch(w / 'last.pt')
        if e is not None and e >= 0:
            got = (f, e + 1)
            print(f'  WARNING: {idx["last"]["file"]} on Drive is damaged; using {f} (epoch {e + 1}) instead')
            break
    if not got:
        return None
    if idx.get('best') and (dr / idx['best']['file']).exists():
        shutil.copyfile(dr / idx['best']['file'], w / 'best.pt')
    if (dr / 'results.csv').exists():
        shutil.copyfile(dr / 'results.csv', local / 'results.csv')
        truncate_results(local / 'results.csv', got[1])
    print(f'  restored {got[0]} (epoch {got[1]}) from {dr}')
    return {'file': got[0], 'epoch': got[1], 'stop': bool(idx['last'].get('stop')) and got[0] == idx['last']['file']}


def read_results(csv_path):
    with open(csv_path, newline='', encoding='utf-8') as f:
        rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f)]
    for r in rows:
        r['fitness'] = 0.1 * float(r['metrics/mAP50(B)']) + 0.9 * float(r['metrics/mAP50-95(B)'])
    return rows


def finalize_run(run_name, local, by_notebook=False):
    """Copies best.pt, last.pt, results.csv, args.yaml to Drive, checks them and writes the Drive marker."""
    from ultralytics.utils.torch_utils import strip_optimizer
    w, dr = local / 'weights', DRUNS / f'{run_name}_app'
    if by_notebook:   # the training loop had finished, but the machine was lost before Ultralytics' final step
        for p in (w / 'last.pt', w / 'best.pt'):
            if p.exists() and ckpt_epoch(p) != -1:
                strip_optimizer(p)
    res = read_results(local / 'results.csv')
    best = max(res, key=lambda r: r['fitness'])
    final = dr / 'final'
    shas = {}
    for name, src in (('best.pt', w / 'best.pt'), ('last.pt', w / 'last.pt'), ('results.csv', local / 'results.csv'),
                      ('args.yaml', local / 'args.yaml')):
        if not src.exists():
            if name == 'args.yaml' and (dr / 'args.yaml').exists():
                shutil.copyfile(dr / 'args.yaml', src)
            else:
                raise PipelineStop(f'{src} is missing after training {run_name}')
        dest, ok = copy_to_drive(src, final / name, verify=True)
        if not ok:
            raise PipelineStop(f'copying {src} to Google Drive failed; run this cell again')
        shas[name] = file_hash(src)
    args = _yaml.safe_load((local / 'args.yaml').read_text(encoding='utf-8'))
    prog = jload(dr / 'progress.json', {}) or {}
    times = list((prog.get('epoch_times_s') or {}).values())
    rows = [check_row(f'args.yaml {k}', v, args.get(k), status='INFO' if DRY_RUN and k in ('epochs', 'imgsz', 'workers')
                      else None) for k, v in EXPECTED_TRAIN_ARGS.items()]
    rows += [check_row('args.yaml data = the app-level dataset', str(DATA_YAML[run_name]), args.get('data')),
             check_row('epochs completed (results.csv rows)', f"{args.get('epochs')} or early stop (patience "
                       f"{args.get('patience')})", len(res), ok=len(res) == args.get('epochs') or
                       len(res) >= int(float(best['epoch'])) + int(args.get('patience', 0)),
                       note=f"epochs in results.csv: {[int(float(r['epoch'])) for r in res][:3]}...{int(float(res[-1]['epoch']))}"),
             check_row('results.csv epochs are 1..n without gaps', list(range(1, len(res) + 1))[-1:],
                       [int(float(r['epoch'])) for r in res][-1:],
                       ok=[int(float(r['epoch'])) for r in res] == list(range(1, len(res) + 1))),
             check_row('best epoch (max fitness 0.1 mAP50 + 0.9 mAP50-95)', 'reported', int(float(best['epoch'])),
                       status='INFO', note=f"val mAP50 {float(best['metrics/mAP50(B)']):.4f}, "
                                           f"mAP50-95 {float(best['metrics/mAP50-95(B)']):.4f} (training-time validation)"),
             check_row('mean time per epoch (min)', 'reported', round(sum(times) / max(1, len(times)) / 60, 2), status='INFO'),
             check_row('final files on Drive (sha256 read back)', 'best.pt, last.pt, results.csv, args.yaml', ', '.join(shas), ok=len(shas) == 4),
             check_row('finished by', 'Ultralytics', 'notebook (after a lost machine)' if by_notebook else 'Ultralytics',
                       status='INFO')]
    set_checks(f'6 {run_name} training', rows)
    hist = load_state('env_history', [])
    info = {'epochs_done': len(res), 'best_epoch': int(float(best['epoch'])),
            'best_val_mAP50_training_time': float(best['metrics/mAP50(B)']),
            'best_val_mAP50_95_training_time': float(best['metrics/mAP50-95(B)']),
            'hours_total': round(sum(times) / 3600, 2), 'mean_epoch_min': round(sum(times) / max(1, len(times)) / 60, 2),
            'gpu': ', '.join(sorted({h.get('gpu') or 'cpu' for h in hist})), 'sha256': shas,
            'drive_final': str(final), 'finalized_by_notebook': by_notebook, 'sessions': prog.get('sessions')}
    dmark(train_key(run_name), info)
    return info


def train_run(run_name):
    if ddone(train_key(run_name)):
        print(f'[skip] {run_name} already trained: {dmarker(train_key(run_name)).get("drive_final")}')
        return
    local, dr = D['runs'] / f'{run_name}_app', DRUNS / f'{run_name}_app'
    dr.mkdir(parents=True, exist_ok=True)
    kill_stale(LAUNCHER, ('python',))   # a training of an earlier, interrupted session
    crash = DRY['crash'].get(run_name) if DRY_RUN else None
    hook = {'drive_run': str(dr), 'drive_mount': str(DD if DRY_RUN else '/content/drive'), 'run': run_name,
            'every': DRIVE_CKPT_EVERY_EPOCHS, 'reserve_gb': DRIVE_RESERVE_GB, 'gpu': env.get('gpu') or 'cpu',
            'n_train': N_TRAIN, 'later_runs': [x for x in RUNS[RUNS.index(run_name) + 1:] if not ddone(train_key(x))],
            'epochs_of': {x: int(RESOLVED[x]['train_kwargs']['epochs']) for x in RUNS},
            'crash': ({'after_epoch': int(crash['after_epoch']), 'code': 98 if crash['mode'] == 'hard' else 97,
                       'flag': str(DS / f'dryrun_crashed_{run_name}')} if crash else None)}
    if (local / 'args.yaml').exists():
        copy_to_drive(local / 'args.yaml', dr / 'args.yaml')
    attempts = 0
    while True:
        last = local / 'weights' / 'last.pt'
        e_local = ckpt_epoch(last) if last.exists() else None
        idx = jload(dr / 'ckpt_index.json', {}) or {}
        e_drive = (idx.get('last') or {}).get('epoch')
        if e_local == -1:   # finished and stripped by Ultralytics
            return finalize_run(run_name, local)
        if e_local is not None and (idx.get('last') or {}).get('stop') and e_drive == e_local + 1:
            print(f'{run_name}: the training loop had finished (final step interrupted); finishing it here')
            return finalize_run(run_name, local, by_notebook=True)
        if e_local is None or (e_drive is not None and e_drive > e_local + 1):
            if e_drive is not None:
                if local.exists():
                    shutil.rmtree(local)
                rest = restore_from_drive(run_name, local)
                if rest and rest['stop']:
                    print(f'{run_name}: the training loop had finished on the earlier machine; finishing it here')
                    return finalize_run(run_name, local, by_notebook=True)
                e_local = None if not rest else rest['epoch'] - 1
            elif local.exists():
                shutil.rmtree(local)   # no completed epoch anywhere: start clean
        sess = jload(dr / 'progress.json', {}) or {}
        sess['sessions'] = sess.get('sessions', 0) + 1
        if not sess.get('epoch_times_s'):
            sess['epoch_times_s'] = {}
        jdump(sess, dr / 'progress.json')
        if e_local is None:
            print(f'\n=== {run_name}: training from {INIT.name} (COCO), {CFG_RUN[run_name].name} ===', flush=True)
            cmd = [sys.executable, LAUNCHER, '--config', CFG_RUN[run_name], '--device', DEVICE]
        else:
            truncate_results(local / 'results.csv', e_local + 1)
            print(f'\n=== {run_name}: resuming after epoch {e_local + 1} from {last} ===', flush=True)
            cmd = [sys.executable, LAUNCHER, '--resume', last, '--device', DEVICE]
        rc, _ = run(cmd, cwd=D['runs'], log=f'train_{run_name}.log', check=False,
                    env=dict(SUBENV, UI2C_HOOK=json.dumps(hook)), progress_every=1 if DRY_RUN else 120)
        if (local / 'args.yaml').exists():
            copy_to_drive(local / 'args.yaml', dr / 'args.yaml')
        if rc == 0:
            if ckpt_epoch(last) == -1:
                return finalize_run(run_name, local)
            raise PipelineStop(f'train.py ended without error, but {last} is not a finished checkpoint')
        if DRY_RUN and rc == 98:
            write_report()
            raise PipelineStop(f'DRY_RUN: simulated loss of the machine during {run_name} (expected in dry run 1)')
        if DRY_RUN and rc == 97:
            print('DRY_RUN: simulated crash; deleting the local run folder so that the resume uses the Drive copy')
            shutil.rmtree(local)
        attempts += 1
        if attempts > MAX_AUTO_RESUME:
            write_report()
            raise PipelineStop(f'training {run_name} stopped {attempts} times (last exit code {rc}); see '
                               f'{D["logs"] / f"train_{run_name}.log"}. Choose Run all to resume from the last checkpoint.')
        print(f'training {run_name} stopped with exit code {rc}; automatic resume {attempts}/{MAX_AUTO_RESUME} in 20 s',
              flush=True)
        time.sleep(0 if DRY_RUN else 20)


def val_metrics(weights, split, half):
    from ultralytics import YOLO
    import ultralytics
    t0 = time.time()
    model = YOLO(str(weights))
    r = model.val(data=str(YOLO_FULL / 'dataset.yaml'), split=split, half=half, device=DEVICE, plots=False,
                  save_json=False, verbose=True, project=str(D['metrics'] / 'ultralytics_runs'),
                  name=f'{Path(weights).parent.parent.name}_{split}_{"fp16" if half else "fp32"}', exist_ok=True,
                  **VAL_SETTINGS)
    box, names = r.box, r.names
    per_class = {names[int(c)]: {'ap50': float(box.ap50[k]), 'ap50_95': float(box.ap[k])}
                 for k, c in enumerate(box.ap_class_index)}
    nt = getattr(r, 'nt_per_class', None)
    return {'split': split, 'variant': 'fp16' if half else 'fp32', 'images': sizes_eval[split],
            'mAP50': float(box.map50), 'mAP50_95': float(box.map), 'precision': float(box.mp), 'recall': float(box.mr),
            'per_class': per_class,
            'instances_per_class': {names[i]: int(n) for i, n in enumerate(nt)} if nt is not None else None,
            'speed_ms_per_image': {k: float(v) for k, v in dict(r.speed).items()}, 'seconds': round(time.time() - t0),
            'settings': {**VAL_SETTINGS, 'half': half, 'device': DEVICE, 'data': str(YOLO_FULL / 'dataset.yaml'),
                         'weights': str(weights), 'weights_sha256': file_hash(weights),
                         'images': 'original ReDraw screenshots (symlinks), resized by Ultralytics as in training'},
            'versions': {'ultralytics': ultralytics.__version__, 'torch': torch.__version__, 'cuda': torch.version.cuda,
                         'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}}


def local_best(run_name):
    """best.pt of a finished run on this machine (copied back from Drive after a machine change)."""
    p = D['runs'] / f'{run_name}_app' / 'weights' / 'best.pt'
    src = DRUNS / f'{run_name}_app' / 'final' / 'best.pt'
    want = dmarker(train_key(run_name)).get('sha256', {}).get('best.pt')
    if not p.exists() or (want and file_hash(p) != want):
        p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, p)
    if want and file_hash(p) != want:
        raise PipelineStop(f'{src} on Drive does not match the SHA-256 recorded after training')
    return p


def evaluate_run(run_name):
    if ddone(eval_key(run_name)):
        print(f'[skip] {run_name} already evaluated')
        return
    w = local_best(run_name)
    variants = [('fp32', False)] + ([('fp16', True)] if torch.cuda.is_available() and ALSO_RUN_FP16_ON_GPU else [])
    res, rows = {}, []
    for split in ('val', 'test'):
        for vname, half in variants:
            print(f'\n--- {run_name}: YOLO.val split={split} {vname} ({sizes_eval[split]:,} images) ---', flush=True)
            m = val_metrics(w, split, half)
            path = D['metrics'] / f'{run_name}_app_{split}_{vname}.json'
            jdump(m, path)
            copy_to_drive(path, DMET / path.name)
            res[f'{split} {vname}'] = m
            rows.append(check_row(f'{split} {vname}: mAP@50 / mAP@50-95', 'reported',
                                  f"{m['mAP50']:.5f} / {m['mAP50_95']:.5f}", status='INFO',
                                  note=f"P {m['precision']:.4f} R {m['recall']:.4f}; {m['seconds']} s"))
            if m.get('instances_per_class'):
                ours = split_counts[split]['per_class_instances']
                same = all(m['instances_per_class'].get(k, 0) == v for k, v in ours.items())
                rows.append(check_row(f'{split} {vname}: Ultralytics instances = label lines', True, same,
                                      status='PASS' if same else 'WARN'))
            rows.append(check_row(f'{split} {vname}: images evaluated', sizes_eval[split], m['images']))
    t = dmarker(train_key(run_name))
    key = 'val fp16' if 'val fp16' in res else 'val fp32'
    rows.append(check_row('val mAP@50: standalone vs training-time best epoch', t.get('best_val_mAP50_training_time'),
                          res[key]['mAP50'], status='INFO',
                          note=f"diff {100 * (res[key]['mAP50'] - (t.get('best_val_mAP50_training_time') or 0)):+.2f} points "
                               f'({key}; training-time validation used the pre-resized copies, FP16)'))
    set_checks(f'6m {run_name} metrics', rows)
    save_state(f'metrics_{run_name}', {'values': {k: {x: v[x] for x in ('mAP50', 'mAP50_95', 'precision', 'recall')}
                                                   for k, v in res.items()},
                                        'files': sorted(str(DMET / f'{run_name}_app_{k.replace(" ", "_")}.json') for k in res)})
    dmark(eval_key(run_name), {'values': {k: v['mAP50'] for k, v in res.items()}})


if training_left() or any(not ddone(eval_key(r)) for r in RUNS):
    sp = load_state('split')
    sizes_eval = {s: sp['sizes'][s] for s in SPLITS}
    split_counts = sp['counts']
    for r in RUNS:
        train_run(r)
        write_report()
        evaluate_run(r)
        write_diagnostics()
        if r in DRY.get('stop_after', []):
            break
else:
    print('[skip] all runs trained and evaluated (Drive markers)')
write_report()

## Step 7 – evaluate_detector and Table 1

`ui2compose-evaluate-detector` (v1.2.0) evaluates both `best.pt` on val and test.
* Its predictions come from `YOLO.val()` with the same settings, and its metrics are computed with `ap.py`.
* It adds percentile bootstrap 95 % intervals over images and over **app groups** (`group_id` of the split manifest, 1,000 resamples).
* It adds the paired difference oversample_5k − baseline.
* Raw predictions are saved (gzip) for later recomputation on the laptop.

`metrics/table1_app_split.md` shows the new numbers next to the published screenshot-split numbers. Those published numbers come from a different test set and are shown only for orientation.

In [ ]:
#@title Step 7 - evaluate_detector (bootstrap CIs over images and app groups, paired difference) and Table 1
import gzip

REF = {'test': {'baseline': jload(CODE / 'results/perclass_ap_results.json')['imgsz_640'],
                'oversample_5k': jload(CODE / 'results/perclass_ap_results.json')['oversample_5k']},
       'val': {'baseline': jload(CODE / 'results/imgsz_640_metrics.json'),
               'oversample_5k': jload(CODE / 'results/oversample_5k_metrics.json')}}


def ref_overall(split, r):
    x = REF[split].get(r) or {}
    return x.get('overall', x)


if any(not ddone(train_key(r)) for r in RUNS):
    print('training is not finished - Step 7 needs all runs')
else:
    if RUN_EVALUATE_DETECTOR and not ddone('evaluate_detector'):
        weights = [local_best(r) for r in RUNS]
        out = {}
        for split in ('val', 'test'):
            dst = D['metrics'] / f'evaluate_detector_{split}.json'
            args = []
            for w in weights:
                args += ['--model', w]
            args += ['--images', YOLO_FULL / 'images' / split, '--labels', YOLO_FULL / 'labels' / split,
                     '--dataset-yaml', YOLO_FULL / 'dataset.yaml', '--groups', SPLIT_MANIFEST, '--bootstrap', BOOTSTRAP,
                     '--imgsz', VAL_SETTINGS['imgsz'], '--conf', VAL_SETTINGS['conf'], '--iou', VAL_SETTINGS['iou'],
                     '--max-det', VAL_SETTINGS['max_det'], '--val-batch', VAL_SETTINGS['batch'], '--device', DEVICE,
                     '--expect-images', load_state('split')['sizes'][split], '--out', dst,
                     '--preds-dir', D['metrics'] / 'preds']
            print(f'\n--- evaluate_detector on {split} (bootstrap {BOOTSTRAP}) ---', flush=True)
            rc, _ = run([sys.executable, '-m', 'ui2compose.evaluation.evaluate_detector', *args],
                        log='evaluate_detector.log', check=False)
            if rc != 0:
                raise PipelineStop(f'evaluate_detector failed on {split} (exit {rc}); log {D["logs"] / "evaluate_detector.log"}')
            copy_to_drive(dst, DMET / dst.name)
            out[split] = str(dst)
        (DMET / 'preds').mkdir(parents=True, exist_ok=True)
        for p in sorted((D['metrics'] / 'preds').glob('*.json')):   # raw predictions, gzipped (laptop recomputation)
            with open(p, 'rb') as fi, gzip.open(DMET / 'preds' / (p.name + '.gz'), 'wb', compresslevel=6) as fo:
                shutil.copyfileobj(fi, fo)
        dmark('evaluate_detector', {'files': out})
    # ---------------- Table 1: app-level split vs the published screenshot-level split ----------------
    T, lines = {}, []
    for split in ('test', 'val'):
        ed = jload(DMET / f'evaluate_detector_{split}.json') if RUN_EVALUATE_DETECTOR else None
        models = {m['name']: m for m in (ed or {}).get('models', [])}
        T[split] = {}
        for r in RUNS:
            v = (load_state(f'metrics_{r}') or {}).get('values', {}).get(f'{split} fp32', {})
            m = models.get(f'{r}_app', {})
            ci_g = (m.get('bootstrap_by_group') or {}).get('mAP50', {})
            ci_i = (m.get('bootstrap') or {}).get('mAP50', {})
            ref = ref_overall(split, r)
            T[split][r] = {'mAP50': v.get('mAP50'), 'mAP50_95': v.get('mAP50_95'), 'precision': v.get('precision'),
                           'recall': v.get('recall'),
                           'evaluate_detector_mAP50': (m.get('metrics') or {}).get('mAP50'),
                           'ci95_mAP50_app_groups': [ci_g.get('lo'), ci_g.get('hi')],
                           'ci95_mAP50_images': [ci_i.get('lo'), ci_i.get('hi')],
                           'published_screenshot_split': {k: ref.get(k) for k in ('mAP50', 'mAP50_95', 'precision', 'recall')},
                           'per_class_ap50': {k: x['ap50'] for k, x in (jload(DMET / f'{r}_app_{split}_fp32.json') or {})
                                              .get('per_class', {}).items()}}
        pair = ((ed or {}).get('paired') or [{}])[0]
        T[split]['paired_diff_mAP50'] = {'other_minus_base': f"{pair.get('other')} - {pair.get('base')}",
                                         'images': (pair.get('diff') or {}).get('mAP50'),
                                         'app_groups': (pair.get('diff_by_group') or {}).get('mAP50')}
    f5 = lambda x: 'n/a' if x is None else f'{100 * x:.2f}'
    md = ['| split | model | mAP@50 (%) | 95 % CI, app groups | mAP@50-95 (%) | P (%) | R (%) | published, screenshot split: mAP@50 / mAP@50-95 |',
          '|---|---|---|---|---|---|---|---|']
    for split in ('test', 'val'):
        for r in RUNS:
            t = T[split][r]
            ci = t['ci95_mAP50_app_groups']
            md.append(f"| {split} | {r} | {f5(t['mAP50'])} | {f5(ci[0])}-{f5(ci[1])} | {f5(t['mAP50_95'])} | "
                      f"{f5(t['precision'])} | {f5(t['recall'])} | {f5(t['published_screenshot_split']['mAP50'])} / "
                      f"{f5(t['published_screenshot_split']['mAP50_95'])} |")
            lines.append(f"TABLE 1 {split} {r}: mAP@50 {f5(t['mAP50'])} % (95 % CI app groups {f5(ci[0])}-{f5(ci[1])}), "
                         f"mAP@50-95 {f5(t['mAP50_95'])} %; published (screenshot split) "
                         f"{f5(t['published_screenshot_split']['mAP50'])} / {f5(t['published_screenshot_split']['mAP50_95'])} %")
        d = T[split]['paired_diff_mAP50'].get('app_groups') or {}
        if d:
            lines.append(f"TABLE 1 {split}: oversample_5k - baseline mAP@50 {f5(d.get('point'))} points "
                         f"(95 % CI app groups {f5(d.get('lo'))} to {f5(d.get('hi'))})")
    names = list(T['test'][RUNS[0]]['per_class_ap50'])
    md += ['', '| class (test AP@50, %) | ' + ' | '.join(RUNS) + ' |', '|---|' + '---|' * len(RUNS)]
    for n in names:
        md.append(f'| {n} | ' + ' | '.join(f5(T['test'][r]['per_class_ap50'].get(n)) for r in RUNS) + ' |')
    note = ('Metrics: YOLO.val() FP32 on the original screenshots of the app-level val/test splits (conf 0.001, NMS IoU 0.7, '
            f"imgsz {VAL_SETTINGS['imgsz']}, max_det 300, batch 16, rect). CIs: ui2compose-evaluate-detector percentile "
            f'bootstrap ({BOOTSTRAP} resamples) over app groups (group_id of the split manifest). "Published" = the '
            'released numbers on the v1.1.0 screenshot-level split (results/*.json); they are NOT comparable as a '
            'measurement of the same test set.')
    (DMET / 'table1_app_split.md').write_text('\n'.join(md) + '\n\n' + note + '\n', encoding='utf-8')
    jdump({'table': T, 'note': note, 'written': now()}, DMET / 'table1_app_split.json')
    save_state('table1', {'lines': lines})
    rows = [check_row(f'{split} {r}: mAP@50 (YOLO.val) = evaluate_detector mAP@50', round(T[split][r]['mAP50'] or 0, 4),
                      round(T[split][r]['evaluate_detector_mAP50'] or 0, 4),
                      status=('PASS' if T[split][r]['evaluate_detector_mAP50'] is not None and
                              abs(T[split][r]['mAP50'] - T[split][r]['evaluate_detector_mAP50']) <= 0.002 else 'WARN')
                      if RUN_EVALUATE_DETECTOR else 'INFO', note='same YOLO.val predictions, metrics by ap.py; can differ only when many confidences are tied '
                           '(e.g. the tiny dry-run model)')
            for split in ('test', 'val') for r in RUNS]
    set_checks('7 evaluate_detector and Table 1', rows)
    print('\n'.join(md) + '\n\n' + note)
    write_report()

## Step 8 – Zip for the laptop (~2.3 GB), copied to Drive with SHA-256

The zip has the same layout as the earlier `ui2compose_redraw_valtest_v1.1.0` package:
* `dataset/`: val + test images (original screenshots), the labels of all three splits, and `dataset.yaml`;
* `raw_xml/`: val + test XMLs and `mapping.csv`;
* `splits/`: manifest, summary, stem lists and fingerprints;
* `weights/<run>_app/`: `best.pt`, `results.csv`, `args.yaml` and epoch times;
* `verification/`: all metrics JSONs, Table 1, raw predictions, and all PASS/FAIL tables;
* `provenance/`: embedded code, configs, logs and pip freeze.

After the copy, the zip on Drive is read back and its SHA-256 is compared with the local one.

In [ ]:
#@title Step 8 - Build the zip for the laptop, copy it to Google Drive (SHA-256 checked)
ZIP_NAME = 'ui2compose_redraw_appsplit_v1.2.0' + ('_DRYRUN' if DRY_RUN else '') + '.zip'
TOP = ZIP_NAME[:-4]

if ddone('package') and (DD / ZIP_NAME).exists():
    pinfo = load_state('package')
    print(f'[skip] zip already on Drive: {DD / ZIP_NAME} ({pinfo.get("size_human")}, sha256 {pinfo.get("sha256")})')
elif any(not ddone(k) for k in [train_key(r) for r in RUNS] + [eval_key(r) for r in RUNS]
                                + (['evaluate_detector'] if RUN_EVALUATE_DETECTOR else [])):
    raise PipelineStop('training/evaluation is not finished - the zip is built after Steps 6-7')
else:
    zip_path = D['out'] / ZIP_NAME
    pkey = 'zip_local'
    if not (done(pkey) and zip_path.exists()):
        rr = raw_root()
        with open(SPLIT_MANIFEST, newline='', encoding='utf-8') as f:
            man = {r['stem']: r for r in csv.DictReader(f)}
        S, Z = zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED
        members = []   # (source path or None, arcname, text or None, compress)
        mapping, problems = [], []
        for s in ('val', 'test'):
            for p in list_images(YOLO_FULL, s):
                src = p.resolve()
                members.append((src, f'dataset/images/{s}/{p.name}', None, S))
                m = STEM_RE.match(p.stem)
                app, n = m.group('app'), m.group('n')
                xml = rr / app / f'hierarchy_{n}.xml'
                shot_md5 = file_hash(src, 'md5')
                if not xml.exists() or shot_md5 != man[p.stem]['md5']:
                    problems.append(p.stem)
                arc_xml = f'raw_xml/{s}/{app}/hierarchy_{n}.xml'
                if xml.exists():
                    members.append((xml, arc_xml, None, Z))
                mapping.append({'split': s, 'stem': p.stem, 'package': app[:-len('-screens')], 'screen_index': n,
                                'group_id': man[p.stem]['group_id'], 'old_split_v1.1.0': man[p.stem]['old_split'],
                                'image_in_zip': f'dataset/images/{s}/{p.name}', 'label_in_zip': f'dataset/labels/{s}/{p.stem}.txt',
                                'xml_in_zip': arc_xml if xml.exists() else '', 'raw_app_dir': app,
                                'raw_hierarchy': f'{app}/hierarchy_{n}.xml', 'raw_screenshot': f'{app}/{src.name}',
                                'image_md5': man[p.stem]['md5'], 'raw_screenshot_md5': shot_md5,
                                'xml_sha256': file_hash(xml) if xml.exists() else ''})
        for s in SPLITS:
            for p in sorted((YOLO_FULL / 'labels' / s).glob('*.txt')):
                members.append((p, f'dataset/labels/{s}/{p.name}', None, Z))
        yaml_text = ('# Portable copy: no "path" key, so Ultralytics uses the folder of this file as the dataset root.\n'
                     '# The app-level split of ui2compose v1.2.0; images/train is not part of this package (labels/train is).\n'
                     'train: images/train\nval: images/val\ntest: images/test\n'
                     f'nc: {len(NAMES)}\nnames:\n' + ''.join(f'  - {n}\n' for n in NAMES))
        members.append((None, 'dataset/dataset.yaml', yaml_text, Z))
        buf = io.StringIO()
        w = csv.DictWriter(buf, fieldnames=list(mapping[0]))
        w.writeheader()
        w.writerows(mapping)
        members.append((None, 'raw_xml/mapping.csv', buf.getvalue(), Z))
        for p in sorted(SPLIT_DIR.iterdir()):   # app_split's unseen_*.txt hold absolute Colab paths: left out
            if p.is_file() and not p.name.startswith('unseen_'):
                arc = (f'splits/{p.name}' if p.name.startswith(('app_split_', 'split_', 'build_dataset_'))
                       else f'splits/audit_of_v1.1.0_screenshot_split/{p.name}')
                members.append((p, arc, None, Z))
        members.append((DRIVE_SPLIT / 'split_fingerprints.json', 'splits/split_fingerprints.json', None, Z))
        members.append((D['data'] / 'resize_sizes.csv', 'splits/resize_sizes.csv', None, Z))
        for p in sorted(DS.glob('*.json')):
            members.append((p, f'verification/state/{p.name}', None, Z))
        for p in sorted(DMET.glob('*')):
            if p.is_file():
                members.append((p, f'verification/metrics/{p.name}', None, Z))
        for p in sorted((DMET / 'preds').glob('*.gz')):
            members.append((p, f'verification/metrics/preds/{p.name}', None, S))
        for r in RUNS:
            fin = DRUNS / f'{r}_app'
            for name in ('best.pt', 'results.csv', 'args.yaml'):
                members.append((fin / 'final' / name, f'weights/{r}_app/{name}', None, Z if name != 'best.pt' else S))
            for name in ('progress.json', 'ckpt_index.json'):
                if (fin / name).exists():
                    members.append((fin / name, f'weights/{r}_app/{name}', None, Z))
            members.append((None, f'weights/{r}_app/train_summary.json', json.dumps(dmarker(train_key(r)), indent=2), Z))
        for p in sorted(CODE.rglob('*')):
            if p.is_file():
                members.append((p, f'provenance/ui2compose_v1.2.0_embedded/{p.relative_to(CODE).as_posix()}', None, Z))
        for p in sorted(D['cfg'].glob('*.yaml')) + [LAUNCHER, D['tools'] / 'resize_worker.py']:
            if p.exists():
                members.append((p, f'provenance/run_files/{p.name}', None, Z))
        for p in sorted(D['logs'].glob('*.log')):
            members.append((p, f'provenance/logs/{p.name}', None, Z))
        freeze = run([sys.executable, '-m', 'pip', 'freeze'], echo=False, quiet=True, check=False)[1]
        members.append((None, 'provenance/pip_freeze.txt', freeze, Z))
        prov = {'written': now(), 'dry_run': DRY_RUN, 'force': FORCE, 'repo_url': REPO_URL, 'repo_tag': REPO_TAG,
                'clone': load_state('clone'), 'pinned': PINNED, 'embedded_sha256': EMBED_SHA256,
                'embedded_tar_sha256': EMBED_TAR_SHA256, 'zenodo': load_state('zenodo'), 'md5_check': marker('md5'),
                'extract': marker('extract'), 'parse': load_state(f'parse_{REPO_TAG}'), 'build': load_state(f'build_{REPO_TAG}'),
                'app_split': marker('appsplit'), 'crosscheck': marker('xcheck'), 'resize': marker('ds_resized'),
                'oversample': marker('ds_os5k'), 'train_setup': load_state('train_setup'),
                'training': {r: dmarker(train_key(r)) for r in RUNS}, 'env_history': load_state('env_history'),
                'settings': {'SEED': SEED, 'NEAR_DUP_DIST': NEAR_DUP_DIST, 'OVERSAMPLE_TARGET': OVERSAMPLE_TARGET,
                             'VAL_SETTINGS': VAL_SETTINGS, 'BOOTSTRAP': BOOTSTRAP, 'CACHE_MODE': CACHE_MODE,
                             'DRIVE_CKPT_EVERY_EPOCHS': DRIVE_CKPT_EVERY_EPOCHS, 'ULTRALYTICS_VERSION': ULTRALYTICS_VERSION},
                'raw_mapping_problems': problems}
        members.append((None, 'provenance/provenance.json', json.dumps(prov, indent=2, default=str), Z))
        sp = load_state('split')
        readme = f'''ui2compose - ReDraw APP-LEVEL split package (SoftwareX revision, reviewer 1: "re-split by app")
written {now()} on Colab{' (DRY RUN on synthetic data)' if DRY_RUN else ''}

The {sum(sp['sizes'].values()):,} labelled ReDraw screens of the v1.1.0 preprocessing (parse_redraw + build_dataset at tag {REPO_TAG}) were
re-split with app_split --resplit (ui2compose v1.2.0): a group is the connected component of "same package OR
identical MD5 OR dHash distance <= {NEAR_DUP_DIST}"; groups go to train/val/test (80/10/10 of the screens) with greedy
multi-label stratification, seed {SEED}. No group, package, MD5 or near-duplicate occurs in two splits (audited).
Screens train/val/test: {sp['sizes']['train']:,} / {sp['sizes']['val']:,} / {sp['sizes']['test']:,}.
YOLOv8s was retrained from COCO yolov8s.pt with configs/train_baseline.yaml and configs/train_oversample_5k.yaml
(rare classes oversampled to ~{OVERSAMPLE_TARGET} instances in the TRAIN split only), hyperparameters unchanged.
Training used copies of the screenshots pre-resized to {TRAIN_IMGSZ} px (long side) with exactly the resize of
Ultralytics' load_image (identical network input, checked); labels are normalised and unchanged.

dataset/            val + test images (original ReDraw screenshots), labels of train/val/test of the NEW split,
                    dataset.yaml (portable: no "path" key)
raw_xml/            raw uiautomator hierarchies of the val + test screens; mapping.csv links every image to its
                    ReDraw app folder, hierarchy_<n>.xml, screenshot_<n>.png, its group_id and its v1.1.0 split
splits/             app_split_manifest.csv (stem, package, group_id, split, old_split, md5, dhash),
                    app_split_summary.json (audit, class support), split_<split>_stems.txt,
                    split_fingerprints.json, build_dataset_split_manifest.csv (cross-check, identical),
                    resize_sizes.csv; audit_of_v1.1.0_screenshot_split/ (app_split on the old split, for reference)
weights/<run>_app/  best.pt, results.csv, args.yaml, progress.json (epoch times), train_summary.json
verification/       metrics/*.json: YOLO.val per run/split (fp32, fp16 on GPU), evaluate_detector_<split>.json
                    (bootstrap CIs over images and app groups, paired difference), table1_app_split.md/.json,
                    preds/ (raw predictions, gzip); state/ (all PASS/FAIL tables: checks.json, environment)
provenance/         embedded v1.2.0 code with SHA-256, run configs, launcher, logs, pip freeze, provenance.json

Examples (laptop, from the unzipped folder):
  yolo val model=weights/oversample_5k_app/best.pt data=dataset/dataset.yaml split=test
  ui2compose-evaluate-detector --model weights/baseline_app/best.pt --model weights/oversample_5k_app/best.pt \\
      --images dataset/images/test --labels dataset/labels/test --dataset-yaml dataset/dataset.yaml \\
      --groups splits/app_split_manifest.csv --bootstrap 1000 --imgsz 640 --out results/eval/detector_test_appsplit.json
'''
        members.append((None, 'README_PACKAGE.txt', readme, Z))
        text, _ = write_report()
        members.append((None, 'REPORT.txt', text, Z))
        tmp = zip_path.with_name(zip_path.name + '.partial')
        t0 = time.time()
        print(f'writing {len(members):,} files into {zip_path} ...', flush=True)
        with zipfile.ZipFile(tmp, 'w', allowZip64=True) as zf:
            for i, (src_p, arc, txt, comp) in enumerate(members, 1):
                if txt is not None:
                    zi = zipfile.ZipInfo(f'{TOP}/{arc}', date_time=time.localtime()[:6])
                    zi.external_attr = 0o644 << 16
                    zf.writestr(zi, txt, compress_type=comp)
                else:
                    zf.write(src_p, f'{TOP}/{arc}', compress_type=comp)
                if i % 5000 == 0:
                    print(f'  {i:,} / {len(members):,}', flush=True)
        with zipfile.ZipFile(tmp) as zf:
            bad = zf.testzip()
        if bad:
            raise PipelineStop(f'zip test failed at {bad}')
        os.replace(tmp, zip_path)
        size = zip_path.stat().st_size
        mark(pkey, {'zip': str(zip_path), 'files': len(members), 'size_bytes': size, 'sha256': file_hash(zip_path),
                    'raw_mapping_problems': len(problems), 'seconds': round(time.time() - t0)})
    zl = marker(pkey)
    pinfo = {'name': ZIP_NAME, 'zip': zl['zip'], 'files': zl['files'], 'size_bytes': zl['size_bytes'],
             'size_human': human(zl['size_bytes']), 'sha256': zl['sha256']}
    cnt = Counter()
    with zipfile.ZipFile(zip_path) as zf:
        for n in zf.namelist():
            parts = n.split('/')[1:]
            cnt['/'.join(parts[:3]) if parts[0] == 'dataset' and len(parts) >= 4 else
                '/'.join(parts[:2]) if parts[0] in ('raw_xml', 'weights') and len(parts) >= 3 else parts[0]] += 1
    sizes_ = load_state('split')['sizes']
    rows = [check_row('val images in zip', sizes_['val'], cnt.get('dataset/images/val', 0)),
            check_row('test images in zip', sizes_['test'], cnt.get('dataset/images/test', 0)),
            check_row('label files in zip (all splits)', sum(sizes_.values()),
                      sum(cnt.get(f'dataset/labels/{s}', 0) for s in SPLITS)),
            check_row('raw XML of val + test in zip', sizes_['val'] + sizes_['test'],
                      cnt.get('raw_xml/val', 0) + cnt.get('raw_xml/test', 0)),
            check_row('val/test images whose raw XML / MD5 do not match', 0, zl['raw_mapping_problems']),
            check_row('best.pt of every run in zip', len(RUNS), sum(1 for r in RUNS if cnt.get(f'weights/{r}_app', 0))),
            check_row('zip size', '< 4 GB (laptop: ~19 GB free)', pinfo['size_human'], ok=pinfo['size_bytes'] < 4e9)]
    # ---- copy to Google Drive ----
    dest = DD / ZIP_NAME
    if not drive_alive():
        mount_drive(force=True)
    dfree = drive_free_gb()
    if dfree is not None and dfree < pinfo['size_bytes'] / 1e9 + 0.05 and not (dest.exists() and dest.stat().st_size == pinfo['size_bytes']):
        write_report()
        raise PipelineStop(f'Google Drive has only {dfree:.2f} GB free, the zip needs {pinfo["size_human"]}. Free up '
                           f'space on drive.google.com (and empty its bin), then run this cell again '
                           f'(the zip is ready at {zip_path}).')
    print(f'copying {pinfo["size_human"]} to {dest} ...', flush=True)
    t0 = time.time()
    dst, ok = copy_to_drive(zip_path, dest, verify=VERIFY_DRIVE_COPY)
    if dst is None:
        raise PipelineStop(f'copying the zip to Google Drive failed; run this cell again (zip ready at {zip_path})')
    (DD / (ZIP_NAME + '.sha256')).write_text(f'{pinfo["sha256"]}  {ZIP_NAME}\n')
    pinfo.update({'drive_path': str(dest), 'drive_sha256_ok': ok, 'copy_seconds': round(time.time() - t0)})
    rows += [check_row('zip on Drive: size', pinfo['size_bytes'], dest.stat().st_size),
             check_row('zip on Drive: SHA-256 read back', True, ok, status=None if VERIFY_DRIVE_COPY else 'INFO')]
    save_state('package', pinfo)
    if set_checks('8 package', rows):
        dmark('package', pinfo)
    write_report()
    print('zip content (files per folder):\n' + '\n'.join(f'  {v:>7,}  {k}' for k, v in sorted(cnt.items())))

## Step 9 – Finished: what to send back

1. **`REPORT.txt`** and **`metrics/table1_app_split.md`** from `MyDrive/ui2compose_revision/retrain_app_split/`.
2. Download **`ui2compose_redraw_appsplit_v1.2.0.zip`** to the laptop and compare it with its `.sha256`. Drive warns that it cannot scan large files; choose *Download anyway*.
3. If anything says FAIL: also send `ui2compose_retrain_diagnostics.zip`.

In [ ]:
#@title Step 9 - Final REPORT.txt, diagnostics, flush Google Drive
write_diagnostics()
text, rep = write_report(final=not remaining_work())
print(text)
left = remaining_work()
banner(('DONE. ' if not left else 'NOT FINISHED: ' + ', '.join(left) + '. ') +
       f'Everything is in {DD}\n  REPORT.txt, {ZIP_NAME} (+ .sha256), metrics/table1_app_split.md, runs/<run>_app/final/',
       ok=not left)
drive_flush()